In [ ]:
import numpy as np
import pandas as pd
import math
import ipywidgets as widgets
import json
import io, os
import shutil
from snsynth import Synthesizer
from snsynth.transform import *
import PIL.Image

import re
from html import escape
from pathlib import Path
import opendp.prelude as dp
dp.enable_features("contrib")

from synthesizers.gauss_copula import copula
from synthesizers.ctgan import wgpctgan # CTGAN trained with Wasserstein loss and gradient penalty
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')
import plotly.figure_factory as ff
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
import copy
import random
from matplotlib.colors import LinearSegmentedColormap

import metrics
import util
import itertools
from scipy import stats, spatial
import shap

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor 
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelBinarizer, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.base import is_classifier, is_regressor 
import umap

from sklearn import ensemble
from sklearn.model_selection import cross_validate, StratifiedKFold

SEED = 8    

def set_global_seed(seed=None):
    seed = SEED if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
def reseed_from_entropy():
    random.seed()
    np.random.seed(None)
    try:
        import torch
        torch.seed()
    except ImportError:
        pass

def seed_info():
    """Text describing how the current run was seeded, for the UI and the reports."""
    if synth and "DP" in synth:
        return (f"<b>Random seed:</b> {SEED} (Evaluation only. DP synthesizer training is intentionally "
                "not seeded, so the DP noise cannot be reproduced)")
    return f"<b>Random seed:</b> {SEED}"

# ----- Data Acquisition -------------------------------------------
# 1. Upload a csv file or use the default dataset for the real data.
    
step1 = "1. Upload a csv file for the Real dataset"
step1_text = "<span style='color: black; font-weight:bold; font-size: 13pt;'>"+step1+"</span>"
step1_label = widgets.HTML(value=step1_text)
display(step1_label)

radio_b = widgets.RadioButtons(
    options=['Demo data', 'Upload File'],
    value=None,
    description='Real Dataset:',
    disabled=False)

output = widgets.Output()
display(radio_b, output)

def on_radio_change(value):
    output.clear_output()      
    global real_df, dataset_name  
    with output:       
        if value['new'] == "Demo data":
            dataset_name = "demo_data"
            default_file = "./datasets/demo_data.csv"
            data_df = pd.read_csv(default_file)            
            real_df = data_df.dropna()            
            display(real_df)          
            generate_metadata(output)
            
        elif value['new'] == "Upload File":            
            uploader = widgets.FileUpload(accept='.csv', multiple=False)            
            display(uploader) 
            uploader.observe(on_upload_change, 'value')            
        
def on_upload_change(value):
    output.clear_output()    
    global real_df, dataset_name    
    with output:
        dataset_name = sanitize_dataset_name(value['new'][0].name)
        file_mess_html = widgets.HTML(value=("<div style='font-size:10pt; margin-top:5px;'>"       
                                             f"<b>Filename:</b> {dataset_name}.csv"
                                             "</div>"))                 
        display(file_mess_html)    
        
        uploaded_file = value['new'][0]        
        data_df = pd.read_csv(io.BytesIO(uploaded_file.content)) 
        null_rows = data_df.isnull().any(axis=1).sum()           
        if null_rows != 0:
            data_df.dropna(inplace=True, ignore_index=True)
            null_mess_html = widgets.HTML(value=("<div style='font-size:10pt; margin-bottom:10px;'>"       
                                          f"A total of <b>{null_rows} row{'s' if null_rows != 1 else ''}</b> containing <b>null values</b> were <b>removed</b> to ensure data completeness."
                                          "</div>"))
            display(null_mess_html)
            
        real_df = data_df
        display(real_df)
        generate_metadata(output)

radio_b.observe(on_radio_change, 'value')


# --------- Data Preparation -----------------------
#2. Generate the real dataset metadata as JSON file.

# Setting the options for sdtype and sdtype properties 
sdtype_opt = [("numerical", 1), ("categorical", 2), ("datetime", 3), ("id", 4)]
prop = ["", "computer_representation", "date_time_format", "order_by", "regex_format"]
prop_options = [" ", "Int64", "Float", "%Y-%m-%d", "[0-9]{10}", "alphabetical", "numerical_value"]
pii_options = ["No", "Yes"]

#Setting the sdtype Dropbox default value
sdtype_opt_dict = {
    "numerical": [1, "computer_representation", "Float"],
    "categorical": [2, "", ""],
    "datetime": [3, "date_time_format", "%Y-%m-%d"],
    "id": [4, "regex_format", ""]
}

def sanitize_dataset_name(filename):
    """Turn an uploaded filename into a safe folder name.
    """
    stem = Path(str(filename)).name   # drop any directory parts
    if stem.lower().endswith(".csv"):
        stem = stem[:-4]
    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", stem).strip("_")
    return stem[:64] or "dataset" # never empty, never '.' or '..'

output1 = widgets.Output()

def ask_seed(output):
    """Ask for the random seed, then continue to the metadata form."""
    with output:
        seed_html = widgets.HTML(
            "<div style='color:black; font-size:11pt; line-height:1.5;'>"
            "<b>Random seed</b> used for the train/test split, non-DP synthesizers and all evaluation steps.<br>"
            "<span>DP synthesizers are intentionally not seeded.</span></div><br>")
        seed_input = widgets.BoundedIntText(value=SEED, min=0, max=2**32 - 1, description='Seed:',
                                            layout=widgets.Layout(width='220px'))
        seed_btn = widgets.Button(description='Set seed', icon='check')
        seed_box = widgets.VBox([seed_html, widgets.HBox([seed_input, seed_btn])])
        display(seed_box)

        def on_seed(b):
            global SEED
            seed_btn.disabled = seed_input.disabled = True  # avoid double clicks
            SEED = int(seed_input.value)
            seed_box.close()
            with output:
                #display(widgets.HTML("<div style='color:black; font-size:11pt;'>"
                                     #f"<b>Random seed:</b> {SEED} </div>"))
                build_metadata_form(output)

        seed_btn.on_click(on_seed)


def generate_metadata(output):
    output1.clear_output()
    global res_dir

    with output:
        res_dir = "./Results/" + dataset_name + "/"

        # New dataset name: create the folder and continue as before
        if not os.path.exists(res_dir):
            os.makedirs(res_dir)
            ask_seed(output)            
            return

        # Folder already exists: ask the user before deleting anything
        confirm_html = widgets.HTML(value=(
            "<div style='font-size:10pt; color:darkred; background-color:LightYellow; padding:5px;'>"
            f"A results folder for <b>{dataset_name}</b> already exists: <b>{res_dir}</b><br>"
            "Continuing will <b>permanently delete</b> all previous results in this folder "
            "(metadata, train/test splits, synthetic data and reports).</div>"))
        yes_btn = widgets.Button(description='Delete and continue', button_style='danger', icon='trash',
                                 layout=widgets.Layout(width='200px'))
        no_btn = widgets.Button(description='Cancel', icon='times')
        confirm_box = widgets.VBox([confirm_html, widgets.HBox([yes_btn, no_btn])])
        display(confirm_box)

        def on_yes(b):
            yes_btn.disabled = no_btn.disabled = True
            confirm_box.close()
            with output:
                # Safety check: only delete a folder directly inside ./Results/
                target = Path(res_dir).resolve()
                if target.parent != Path("./Results").resolve():
                    display(widgets.HTML(f"<span style='color:darkred;'>Refusing to delete unsafe path: {target}</span>"))
                    return
                try:
                    shutil.rmtree(target) # no ignore_errors: failures are reported
                    os.makedirs(res_dir)
                except OSError as e:
                    display(widgets.HTML(f"<span style='color:darkred;'>Could not delete {res_dir}: {e}<br>"
                                         "Close any open files from this folder and try again.</span>"))
                    return
                ask_seed(output)

        def on_no(b):
            yes_btn.disabled = no_btn.disabled = True
            confirm_box.close()
            with output:
                cancel_html = widgets.HTML(value=(
                    "<div style='font-size:10pt; color:#800080;'>"
                    "Deletion cancelled. Previous results were kept.<br>"
                    "To run the pipeline again on the same CSV file, rename the existing results folder "
                    f"<b>{res_dir}</b> (for example to <b>Results/{dataset_name}_run1/</b>) "
                    "using the JupyterLab file browser, then click <b>Continue</b>.</div>"))
                continue_btn = widgets.Button(description='Continue', icon='check')
                cancel_box = widgets.VBox([cancel_html, continue_btn])
                display(cancel_box)

                def on_continue(b):
                    continue_btn.disabled = True
                    cancel_box.close()
                    generate_metadata(output)   # checks the folder again

                continue_btn.on_click(on_continue)

        yes_btn.on_click(on_yes)
        no_btn.on_click(on_no)


def build_metadata_form(output):
   
    global metadata_dict
    with output:
        step2 = "2. Generate the real dataset metadata as JSON file"
        display(widgets.HTML(f"<span style='color: black; font-weight:bold; font-size: 13pt;'>{step2}</span>"))

        # Generate metadata with default property values
        metadata = util.infer_metadata(real_df)
        metadata_dict = util.modify_generate_metadata(real_df, metadata, verbose=False)

        # Build auto_sdtype for UI form
        auto_sdtype = []
        for col, col_info in metadata_dict["columns"].items():
            sdtype = col_info["sdtype"]

            # Default property and value
            if sdtype == "categorical":
                prop_name = ""
                prop_value = ""
            else:
                prop_name = sdtype_opt_dict[sdtype][1]
                prop_value = sdtype_opt_dict[sdtype][2]
                if prop_name and prop_name in col_info:
                    prop_value = col_info[prop_name]

            auto_sdtype.append([
                col,
                sdtype_opt_dict[sdtype][0],  # numeric code for dropdown
                prop_name,
                prop_value,
                pii_options[0]
            ])

        display_metadata_form(output, metadata_dict, auto_sdtype)
        
def display_metadata_form(output, metadata, auto_sdtype):
    global form, json_filename
    with output:        
        form_item_layout = widgets.Layout(display='flex', flex_flow='row', justify_content='space-between', width='100%')        
        form_rows= [widgets.HBox([widgets.Label(value="Column"), widgets.Label(value="Sdtype"), 
                                  widgets.Label(value="Sdtype Property"), widgets.Label(value="Property Value"), 
                                  widgets.Label(value="Quasi-identifier")], 
                                  layout= form_item_layout)]        
        hbox_items = [widgets.HBox([widgets.Label(value=str(k), layout= widgets.Layout(width='30%', height='40px')), 
                      widgets.Dropdown(options=sdtype_opt, value=v), widgets.Dropdown(options=prop, value=o), 
                      widgets.Combobox(options=prop_options, placeholder=p, value=""), widgets.Dropdown(options=pii_options, value=i)], 
                      layout=form_item_layout) for k, v, o, p, i in auto_sdtype]
        
        form_rows.extend(hbox_items)        
        form = widgets.GridBox(form_rows, layout=widgets.Layout(display='flex', flex_flow='column', align_content = "space-between", width='75%'))
        # Update metadata        
        json_filename = dataset_name+"_metadata.json" 
        button_save = widgets.Button(
            description='Save',
            disabled=False,
            tooltip='Save Metadata',
            icon='check')        
        hbox_save = widgets.HBox([button_save], layout=widgets.Layout(width='100%')) 
        display(form, hbox_save, output1)
        button_save.on_click(save_metadata)

def save_metadata(result):
    output1.clear_output()
    global const_metadata_updated, train_real_df, test_real_df, train_filename, test_filename

    with output1:
        # Initialize metadata dictionaries
        metadata_updated = {"columns": {}}
        const_metadata_updated = {"columns": {}}

        column = {}
        const_column = {}

        # Loop through the form rows
        for val in form.children[1:]:
            column_name = val.children[0].value
            keys = [*sdtype_opt_dict]
            sdtype_option = keys[val.children[1].value - 1]

            column_entry = {"sdtype": sdtype_option}
            const_column_entry = {"sdtype": sdtype_option, "pii": val.children[4].value}

            # Add property only if non-empty
            sdtype_property = val.children[2].value
            property_value = val.children[3].value or val.children[3].placeholder
            if sdtype_property and property_value:
                column_entry[sdtype_property] = property_value
                const_column_entry[sdtype_property] = property_value

            column[column_name] = column_entry
            const_column[column_name] = const_column_entry

        metadata_updated["columns"] = column
        const_metadata_updated["columns"] = const_column
        metadata_updated["seed"] = SEED           
        const_metadata_updated["seed"] = SEED   

        # Save metadata to JSON
        json_filename = dataset_name + "_metadata.json"
        if not os.path.exists(res_dir):
            os.makedirs(res_dir)
        with open(os.path.join(res_dir, json_filename), "w") as f:
            json.dump(metadata_updated, f, indent=4)

        # Display success message
        message = f"Metadata saved in {res_dir}{json_filename}"
        display(widgets.HTML(f"<span style='color: #800080; font-weight:bold; font-size: 10pt;'>{message}</span>"))

        # Split dataset into train/test
        train_real_df, test_real_df = train_test_split(real_df, test_size=0.5, random_state=SEED) 
        train_filename = os.path.join(res_dir, "train_" + dataset_name + ".csv")
        test_filename = os.path.join(res_dir, "test_" + dataset_name + ".csv")
        train_real_df.to_csv(train_filename, index=False)
        test_real_df.to_csv(test_filename, index=False)
       
        message = "Real dataset split 50/50 into train and test sets."
        path_mess = "The train and test datasets are saved in: " +res_dir
        mess_text = "<span style='color: black; font-weight:bold; font-size: 11pt;'>"+message+"</span><br>"+"<span style='color: #800080; font-weight:bold; font-size: 10pt;'>"+path_mess+"</span>"
        mess_html = widgets.HTML(value=mess_text)        
        display(mess_html)

    add_constraint(output1)

# Add constraints to columns, such as providing min/max boundaries for numerical columns. 
output2 = widgets.Output()
def add_constraint(output1): 
    output2.clear_output()
    with output1:
        step3 = "For generating synthetic data with marginal methods, continuous variables need to be discretized."
        step3_text = "<span style='color: black; font-weight:bold; font-size: 11pt;'>"+step3+"</span>"
        step3_html = widgets.HTML(value=step3_text)
        display(step3_html)      
       
        choose_label = widgets.Label(value='Please choose:')        
        range_radio = widgets.RadioButtons(options=[ 
                                                'Provide min/max boundaries.', 
                                                'Min/max from training set (may pose privacy risk)',
                                                'Compute min/max boundaries from the DP cumulative distribution.',
                                                'All categorical.'    
                                            ],
                                            value=None,
                                            layout={'width': 'max-content'})
        
        display(choose_label, range_radio, output2)
        range_radio.observe(on_range_radio_change, 'value')      

def on_range_radio_change(value): 
    output2.clear_output()
    global  bounds_form, metadata_obj_dict, cont_variables, dp_cont_variables, pii_cont_variables, used_budget, ep_boundedFloat, public_bounds_form    
    with output2:
        # Update metadata constraints 
        metadata_obj_dict = copy.deepcopy(const_metadata_updated)
        used_budget = (0, 0) 
        cont_variables = [] 
        dp_cont_variables = []
        for var in metadata_obj_dict["columns"].keys():            
                if metadata_obj_dict["columns"][var]["sdtype"] == "numerical":
                    var_min = train_real_df[var].min()  
                    var_max = train_real_df[var].max()
                    cont_variables.append([var, var_min, var_max])
                    metadata_obj_dict["columns"][var]["range"] = [float(var_min), float(var_max)]                    
                elif metadata_obj_dict["columns"][var]["sdtype"] == "categorical":                
                    var_cat_range = list(np.unique(train_real_df[var]).astype(str))
                    metadata_obj_dict["columns"][var]["domain"] = var_cat_range     
              
        pii_cont_variables = [col for col in cont_variables
                              if metadata_obj_dict["columns"][col[0]]["pii"] == 'Yes'] 
        
        if value['new'] == 'Provide min/max boundaries.':
             if len(cont_variables) != 0:
                if pii_cont_variables:
                    variables = pii_cont_variables
                    header = ("Provide min/max boundaries for those continuous variables "
                              "marked as personal information (Quasi-identifier=Yes).")
                else:
                    variables = cont_variables
                    header = "Provide min/max boundaries for continuous variables."

                mess_html = ("<div style='font-size:11pt; line-height:1.5;'>"
                             f"<span style='color:black; font-size:11pt;'> <b>{header}</b></span><br>"
                             "<span style='color:darkred; background-color:LightYellow; font-size:10pt;'> Boundaries from the training set are displayed by default."
                             " If left unchanged, they may pose a privacy risk.</span><br>"
                             "</div><br>")
                display(widgets.HTML(value=mess_html))  
                
                form_layout = widgets.Layout(display='flex', flex_flow='row', justify_content='space-between', width='90%')
                bounds_items = [widgets.HBox([widgets.Label(value=str(k[0]), layout= widgets.Layout(width='15%', height='40px')), 
                                              widgets.widgets.FloatText(value=k[1], description='min:'), widgets.widgets.FloatText(value=k[2], description='max:')], 
                                              layout=form_layout) for k in variables]
                
                bounds_form = widgets.GridBox(bounds_items, layout=widgets.Layout(
                                        display='flex',
                                        flex_flow='column',
                                        align_content = "space-between",
                                        width='75%'
                                    ))
                button_submit = widgets.Button(description='Save', tooltip='Save min/max values', icon='check')
                
                display(bounds_form, button_submit)                
                button_submit.on_click(save_min_max_constraint) 
                 
        elif value['new'] == 'Min/max from training set (may pose privacy risk)':
             if len(cont_variables) != 0:
                cont_variables = []                
                for var in metadata_obj_dict["columns"].keys():            
                        if metadata_obj_dict["columns"][var]["sdtype"] == "numerical":
                            var_min = train_real_df[var].min()
                            var_max = train_real_df[var].max()
                            cont_variables.append([var, var_min, var_max])
                            metadata_obj_dict["columns"][var]["range"] = [float(var_min), float(var_max)]
                         
                json_filename = "_metadata_constraints.json"                 
                with open(res_dir+dataset_name+json_filename, "w") as outfile: 
                    json.dump(metadata_obj_dict, outfile)
                    message = "Metadata and boundaries constraints are saved in: " +res_dir+dataset_name+json_filename
                    mess_text = "<span style='color: #800080; font-weight:bold; font-size: 10pt;'>"+message+"</span>"
                    mess_html = widgets.HTML(value=mess_text)               
                    display(mess_html) 
                choose_synthesizer(output2)   
                 
        elif value['new'] == 'Compute min/max boundaries from the DP cumulative distribution.':
            if len(cont_variables) != 0:
                # privacy settings                
                dp_message = "DP will be applied to the discretization of continuous variables marked as personal information (Quasi-identifier=Yes)."
                dp_mess_text = "<span style='color: black; font-weight:bold; font-size: 11pt;'>"+dp_message+"</span>"
                dp_mess_html = widgets.HTML(value=dp_mess_text) 
                if pii_cont_variables:
                    pb_html = widgets.HTML(value=(
                        "<div style='font-size:11pt; line-height:1.5;'>"
                        "<b>Enter public min/max bounds for each variable.</b><br>"
                        "<span style='color:darkred; font-size:10pt; background-color:LightYellow;'>"
                        "Use domain knowledge. "
                        "Do NOT look at the data. Values outside the bounds will be clipped.</span>"
                        "</div><br>"))
                    form_layout = widgets.Layout(display='flex', flex_flow='row', justify_content='space-between', width='90%')
                    pb_items = [widgets.HBox([widgets.Label(value=str(k[0]), layout=widgets.Layout(width='15%', height='40px')),
                                              widgets.FloatText(value=0.0, description='Public min:'),
                                              widgets.FloatText(value=0.0, description='Public max:')],
                                             layout=form_layout) for k in pii_cont_variables]
                    public_bounds_form = widgets.GridBox(pb_items, layout=widgets.Layout(
                                            display='flex', flex_flow='column', align_content='space-between', width='75%'))
                    display(pb_html, public_bounds_form)

                # Choose the privacy budget (epsilon)
                ep_label = widgets.Label(value=r'Choose Privacy Budget ($\epsilon$) for computing DP distributions:')
                ep_boundedFloat = widgets.BoundedFloatText(value=0.1, min=0.01, max=15.0, step=0.01, disabled=False)
                ep_button = widgets.Button(description='Submit', disabled=False, button_style='', tooltip='submit', icon='check')
                hbox_ep = widgets.HBox([ep_label, ep_boundedFloat, ep_button], layout=widgets.Layout(width='100%')) 
                display(hbox_ep)
                ep_button.on_click(dp_ep_button_on_click)               
                
        elif value['new'] == 'All categorical.':
            if len(cont_variables) != 0:
                print("Not all variables are categorical. Continuous variables need to be discretized.")
            else:               
                display(widgets.HTML("<span style='color:black; font-weight:bold; font-size:10pt;'>"
                                     "No continuous variables: discretization is not needed.</span>"))                
                json_filename = "_metadata_constraints.json"
                with open(res_dir+dataset_name+json_filename, "w") as outfile: 
                    json.dump(metadata_obj_dict, outfile)
                    message = "Metadata and boundaries constraints are saved in: " +res_dir+dataset_name+json_filename
                    mess_text = "<span style='color: #800080; font-weight:bold; font-size: 10pt;'>"+message+"</span>"
                    mess_html = widgets.HTML(value=mess_text)               
                    display(mess_html) 
                choose_synthesizer(output2)      

def dp_ep_button_on_click(result):
    global bounds_form, used_budget, dp_cont_variables, budget_per_var
    
    if not pii_cont_variables:
        output2.clear_output()
        with output2:
            message_1 = "Non continuous variable marked as personal information (Quasi-identifier=Yes)."
            message_2 = "Discretizations of continuous variables will be applied using the train (real) min/max."
            display(widgets.HTML(
                f"<span style='color: #800080; font-weight:bold; font-size:10pt;line-height:1.5;'>{message_1}</span><br>"
                f"<span style='color:darkred; background-color:LightYellow; font-size:10pt;'>{message_2}</span>"))
            choose_synthesizer(output2)
        return

    # Read and validate public bounds before clearing the form
    public_bounds = {}
    for row in public_bounds_form.children:
        name, lo, hi = row.children[0].value, row.children[1].value, row.children[2].value
        if not lo < hi:
            with output2:
                display(widgets.HTML(f"<span style='color:darkred;'>Invalid public bounds for "
                                     f"<b>{name}</b>: min must be smaller than max.</span>"))
            return
        public_bounds[name] = (lo, hi)

    output2.clear_output()
    with output2:
        epsilon_bounds = ep_boundedFloat.value
        used_budget = (0, 0)
        dp_cont_variables = []

        num_cont = len(pii_cont_variables)
        epsilon_per_query = epsilon_bounds / num_cont
        budget_per_var = {}  
       
        for col in pii_cont_variables:
            lower, upper = public_bounds[col[0]]
            var = np.clip(train_real_df[col[0]].to_numpy().astype(np.float64), lower, upper)
            edges = np.linspace(lower, upper, num=100)

            meas = make_distribution_dp_counts(edges, epsilon_per_query)
            eps_i = float(meas.map(d_in=1))
            counts = meas(var)
            budget_per_var[col[0]] = eps_i

            bin_counts = np.clip(np.asarray(counts, dtype=float), 0, None)
            total = bin_counts.sum()
            if total == 0:
                low, high = lower, upper
            else:
                cum = np.cumsum(bin_counts) / total
                b_lo = int(np.searchsorted(cum, 0.025))
                b_hi = int(np.searchsorted(cum, 0.975))
                low = edges[max(b_lo - 1, 0)]
                high = edges[min(b_hi, len(edges) - 1)]
            dp_cont_variables.append([col[0], np.round(low, decimals=1), np.round(high, decimals=1)])

        used_budget = (float(sum(budget_per_var.values())), 0.0)
        
        ep_spent_label = widgets.Label(value=r'Used privacy budget ($\epsilon$, $\delta$) : ' + f"({used_budget[0]:.4g}, 0)")

        form_layout = widgets.Layout(display='flex', flex_flow='row', justify_content='space-between', width='90%')
        bounds_items = [widgets.HBox([widgets.Label(value=str(k[0]), layout=widgets.Layout(width='15%', height='40px')),
                                      widgets.FloatText(value=k[1], description='DP min:'),
                                      widgets.FloatText(value=k[2], description='DP max:')],
                                     layout=form_layout) for k in dp_cont_variables]
        bounds_form = widgets.GridBox(bounds_items, layout=widgets.Layout(
                                display='flex', flex_flow='column', align_content="space-between", width='75%'))
        button_submit = widgets.Button(description='Submit', tooltip='Submit DP min/max values', icon='check')
        display(ep_spent_label, bounds_form, button_submit)
        button_submit.on_click(save_min_max_constraint) 
                  

def make_distribution_dp_counts(edges, epsilon):
    """epsilon-DP noisy histogram over public edges (neighbors = add/remove one record)."""
    edges = [float(e) for e in edges]
    bin_names = [str(i) for i in range(len(edges) + 1)]
    space = dp.vector_domain(dp.atom_domain(T=float)), dp.symmetric_distance()
    meas = (
        space >>
        dp.t.then_find_bin(edges=edges) >>
        dp.t.then_index(bin_names, "0") >>
        dp.t.then_count_by_categories(categories=bin_names, null_category=False) >>
        dp.m.then_laplace(scale=1.0 / epsilon)
    )
    actual_eps = meas.map(d_in=1)
    if actual_eps > epsilon * (1 + 1e-9):
        raise ValueError(f"DP histogram spends ε={actual_eps}, more than the requested {epsilon}")
    return meas  

def save_min_max_constraint(result):
    output2.clear_output()
    global metadata_obj_dict
    with output2:        
        for val in bounds_form.children[0:]:
            if len(dp_cont_variables) != 0:
               name = val.children[0].value
               metadata_obj_dict["columns"][name]['dp_range'] = [val.children[1].value, val.children[2].value]
               metadata_obj_dict["columns"][name]['budget_used'] = [budget_per_var[name], 0.0]
            else:
               metadata_obj_dict["columns"][val.children[0].value]['range'] = [val.children[1].value, val.children[2].value]
                
        json_filename = "_metadata_constraints.json"
        with open(res_dir+dataset_name+json_filename, "w") as outfile:           
            json.dump(metadata_obj_dict, outfile)
            message = "Metadata and boundaries constraints are saved in: " +res_dir+dataset_name+json_filename
            mess_text = "<span style='color: #800080; font-weight:bold; font-size: 10pt;'>"+message+"</span>"
            mess_html = widgets.HTML(value=mess_text)               
            display(mess_html) 
        choose_synthesizer(output2)
        

#  ------------- Modeling ----------------------------------------
# 3. Choose and train a Synthesizer to generate the synthetic data.

# A set of functions to define variable transformation before sythetization
def ScalarRange_constraint(column_name, low_value, high_value, strict_boundaries=False):
    constraint = {
                    'constraint_class': 'ScalarRange',    
                    'constraint_parameters': {
                        'column_name': column_name,
                        'low_value': low_value,
                        'high_value': high_value,
                        'strict_boundaries': strict_boundaries
                    }
                }
    return constraint
    
# Transform variables for Marginal or Statistical methods
def add_trans_num(cont_vars, bins=30):
    trans = []
    for n in cont_vars:        
        trans.append([n[0], BinTransformer(lower=n[1][0], upper=n[1][1], bins=bins)])
    return trans

def add_trans_cat(cat_vars, nullable=False): # Categorical variables (nominal and ordinal) 
    trans = []
    for n in cat_vars:
        trans.append([n[0], LabelTransformer(nullable=nullable)])    
    return trans

def add_trans_drop(other_vars):
    trans = []
    for n in other_vars:
        trans.append([n[0], DropTransformer()])
    return trans

# Transform variables for GAN methods 
def add_gan_trans_num(cont_vars, train_data): # Continuous variables
    trans = []
    for n in cont_vars:        
        trans.append([n[0], MinMaxTransformer(lower=min(train_data[n[0]]), upper=max(train_data[n[0]]))]) # Variable min/max from the training dataset       
    return trans

def add_gan_trans_cat(cat_vars, nullable=False): # Categorical variables (nominal and ordinal)
    trans = []
    for n in cat_vars:
        trans.append([n[0], ChainTransformer([LabelTransformer(nullable=nullable), OneHotEncoder()])])    
    return trans

# Transform numerical variables for Differentially Pivate GAN methods
def add_dp_gan_trans_num(cont_vars):  
    trans = []
    for n in cont_vars:        
        trans.append([n[0], MinMaxTransformer(lower=n[1][0], upper=n[1][1])])        
    return trans

output3 = widgets.Output()
def choose_synthesizer(output2):
    output3.clear_output()   
    with output2:       
        step4 = "3. Choose and train a Synthesizer to generate the synthetic data."
        step4_text = "<span style='color: black; font-weight:bold; font-size: 13pt;'>"+step4+"</span>"
        step4_html = widgets.HTML(value=step4_text)
        print()
        display(step4_html)
        
        synthesizer = widgets.Select(options=['Baseline: Identity', 'GaussianCopula', 'WGP: CTGAN', 'DP: AIM', 'DP: MWEM', 'DP: PATE-CTGAN'],
                                     value=None,   
                                     description='Synthesizer:',
                                     disabled=False,
                                    layout=widgets.Layout(width='300px', height='130px'))
        display(synthesizer, output3)
        synthesizer.observe(synth_on_change, 'value')

def synth_on_change(value):
    output3.clear_output()
    global synth
    with output3:
        synth = value["new"]
        train_button = widgets.Button(description='Train Synthesizer', disabled=False, button_style='', tooltip='Train a synthesizer', icon='check')
        display(train_button)
        train_button.on_click(train_on_click)

def save_and_show_synthetic(df, filename, title):
    """Save the synthetic data, display it and open the evaluation step (all synthesizers)."""
    print(title, "\n")
    path = res_dir + filename + ".csv"
    df.to_csv(path, index=False)
    display(df, widgets.HTML("<span style='color: #800080; font-weight:bold; font-size: 10pt;'>"
                             f"The synthetic dataset is saved in: {path}</span>"))
    add_diagnostic(output3)

def standard_delta(n):
    """δ ≤ 1/n², rounded down to a power of ten, 
    n is the number of training records.
    """
    n = max(int(n), 2)
    return 10.0 ** -math.ceil(2 * math.log10(n))

# UI name: (SmartNoise synthesizer name, output file prefix, uses delta)
DP_SYNTHESIZERS = {
    "DP: AIM":        ("aim",       "AIM_DP_synthetic_ep_",       True),
    "DP: MWEM":       ("mwem",      "MWEM_DP_synthetic_ep_",      False),   # pure ε-DP
    "DP: PATE-CTGAN": ("patectgan", "DP_PATECTGAN_synthetic_ep_", True),
}

def print_dp_training_info():
    """Print the bounds and privacy budget, and set the total epsilon/delta."""
    global total_ep, total_delta, synth_delta
    if dp_cont_var:
        print("Continuous variables (min, max) boundaries used in preprocessing (e.g., discretization)")
        for n, v in dp_cont_var:
            print(f"{n}, ({v[0]}, {v[1]})")
    print()
    print("Training ", synth)

    ep = synt_ep_boundedFloat.value
    uses_delta = DP_SYNTHESIZERS[synth][2]
    synth_delta = standard_delta(len(train_real_df)) if uses_delta else 0.0

    total_ep = round(used_budget[0] + ep, 6)
    total_delta = used_budget[1] + synth_delta     # bounds step is pure ε-DP, so used_budget[1] = 0

    print(f"You selected epsilon = {ep}")
    if uses_delta:
        print(f"Delta = {synth_delta:.0e} (≤ 1/n², n = {len(train_real_df)} training records)")
    else:
        print("Delta = 0 (MWEM is pure ε-DP)")
    print(f"Total privacy budget used: ε = {total_ep}, δ = {total_delta:.0e}")
    return ep

def fit_dp_synthesizer(ep, **extra_kwargs):
    """Train the selected DP synthesizer (not seeded on purpose) and save its output."""
    global synthetic_data
    reseed_from_entropy()
    sn_name, prefix, uses_delta = DP_SYNTHESIZERS[synth]
    if uses_delta:
        extra_kwargs["delta"] = synth_delta        # MWEM has no delta parameter
    dp_synth = Synthesizer.create(sn_name, epsilon=ep, verbose=True, **extra_kwargs)
    synthetic_data = dp_synth.fit_sample(train_real_df, transformer=tt,
                                         categorical_columns=cat_cols,
                                         ordinal_columns=ord_cols,
                                         continuous_columns=cont_cols)
    save_and_show_synthetic(synthetic_data, prefix + str(total_ep) + "_" + dataset_name,
                            f"{synth} Synthetic Data:")

def train_on_click(b): 
    output3.clear_output()
    global tt, cont_var, dp_cont_var, ord_var, cat_var, other_var, cont_cols, ord_cols, cat_cols, other_cols, synth, synthetic_data     
    with output3:        
        cont_var, dp_cont_var, ord_var, cat_var, other_var = [], [], [], [], []
        cont_cols, ord_cols, cat_cols, other_cols = [], [], [], []
        tt = []
        for var, val in metadata_obj_dict["columns"].items():
            if 'range' in val.keys():
                cont_var.append([var, metadata_obj_dict["columns"][var]["range"]])
                if 'dp_range' in val.keys():
                    dp_cont_var.append([var, metadata_obj_dict["columns"][var]["dp_range"]])
                else:
                    dp_cont_var.append([var, metadata_obj_dict["columns"][var]["range"]])
                    
            elif 'order_by' in val.keys():                
                ord_var.append([var, metadata_obj_dict["columns"][var]["domain"]])               
            elif 'domain' in val.keys():
                cat_var.append([var, metadata_obj_dict["columns"][var]["domain"]])
            else:
                other_var.append([var, metadata_obj_dict["columns"][var]["sdtype"]])
                
        if len(cont_var) !=0:
                cont_cols = list(np.array(cont_var, dtype='object')[:, 0])                    
        if len(ord_var) !=0:
                ord_cols = list(np.array(ord_var, dtype='object')[:, 0]) 
        if len(cat_var) !=0:
                cat_cols = list(np.array(cat_var, dtype='object')[:, 0])                    
        if len(other_var) !=0:               
                other_cols = list(np.array(other_var, dtype='object')[:, 0])

        message = "50% of the Real dataset is used to train the synthesizer."
        path_mess = "The Train (Real) dataset used is in: " +train_filename
        mess_text = "<span style='color: black; font-weight:bold; font-size: 11pt;'>"+message+"</span><br>"+"<span style='color: #800080; font-weight:bold; font-size: 10pt;'>"+path_mess+"</span>"
        mess_html = widgets.HTML(value=mess_text)        
        display(mess_html)
                                   
        if synth == 'Baseline: Identity': 
            # Datetime/id columns are not modelled (same as GaussianCopula and the DP synthesizers)
            synthetic_data = train_real_df.drop(columns=other_cols, errors='ignore').copy()
            display(widgets.HTML("<span style='color:black; font-size:11pt;'>"
                                 "The generated Synthetic dataset is a copy of the Train (Real) dataset.</span>"))
            save_and_show_synthetic(synthetic_data, "Identity_" + dataset_name, "Baseline: Identity")
            
        elif synth == 'GaussianCopula':
            set_global_seed()
            bounds = {name: tuple(rng) for name, rng in cont_var}
            print("Continuous variable bounds (used to clip synthetic values):")
            for name, rng in bounds.items():
                print(f"{name}, {rng}")
                
            
            gscopula = copula.GaussianCopulaSynthesizer(categorical_columns=cat_cols, 
                                                        ordinal_columns=ord_cols,
                                                        continuous_columns=cont_cols,
                                                        bounds=bounds,
                                                        random_state= SEED)
            
            gscopula.fit(train_real_df.drop(columns=other_cols, errors='ignore'))
            synthetic_data = gscopula.sample(train_real_df.shape[0])
            save_and_show_synthetic(synthetic_data, "GaussianCopula_" + dataset_name, "GaussianCopula Synthetic Data:")          
                      
            
        elif synth == "WGP: CTGAN":
            
            bounds = {name: (float(rng[0]), float(rng[1])) for name, rng in cont_var}

            print("Continuous variable bounds (synthetic values are clipped to these):")
            for name, (lo, hi) in bounds.items():
                print(f"{name}, ({lo:g}, {hi:g})")

            train_model_df = train_real_df.drop(columns=other_cols, errors='ignore')
            set_global_seed()
            wctgan = wgpctgan.CTGANSynthesizer(epochs=500, batch_size=100, critic_steps=5, random_state=SEED)
            wctgan.fit(train_model_df,
                       continuous_columns=cont_cols,
                       categorical_columns=cat_cols,
                       ordinal_columns=ord_cols,
                       bounds=bounds)
            synthetic_data = wctgan.sample(train_model_df.shape[0])
            save_and_show_synthetic(synthetic_data, "WGP_CTGAN_" + dataset_name, "WGP: CTGAN Synthetic Data:")   
                
        elif "DP" in synth:
            
            global synt_ep_boundedFloat
                        
            trans_cont, trans_ord, trans_cat, trans_other = [], [], [], []

            if "GAN" in synth:                
                if len(dp_cont_var) !=0:
                    trans_cont = add_dp_gan_trans_num(dp_cont_var)                    
                if len(ord_var) !=0:                    
                    trans_ord = add_gan_trans_cat(ord_var) # In Smartnoise nominal and ordinal variables are transformed using LabelTransformer and OneHotEncoded                
                if len(cat_var) !=0:                    
                    trans_cat = add_gan_trans_cat(cat_var)                    
                if len(other_var) !=0:
                    trans_other = add_trans_drop(other_var)
                                
                trans = trans_cont + trans_cat + trans_ord + trans_other
                
                constraints = {}
                for col in trans:                
                    constraints[col[0]]=col[1]
               
                tt = TableTransformer.create(train_real_df, style='gan', nullable=False, categorical_columns=cat_cols, 
                                             ordinal_columns=ord_cols, continuous_columns=cont_cols,
                                             constraints = constraints)               
            else:
                
                if len(dp_cont_var) !=0:
                    trans_cont = add_trans_num(dp_cont_var)
                if len(ord_var) !=0:
                    trans_ord = add_trans_cat(ord_var)
                if len(cat_var) !=0:                    
                    trans_cat = add_trans_cat(cat_var)
                if len(other_var) !=0: 
                    trans_other = add_trans_drop(other_var)
                                
                trans = trans_cont + trans_cat + trans_ord + trans_other
                
                constraints = {}
                for col in trans:                
                    constraints[col[0]]=col[1]
               
                tt = TableTransformer.create(train_real_df, style='cube', 
                                             nullable=False,
                                             categorical_columns=cat_cols,
                                             ordinal_columns=ord_cols,
                                             continuous_columns=cont_cols,
                                             constraints = constraints)
                        
            # Choose the privacy budget (epsilon)
            message = "Note that the privacy budget for discretizing Quasi-identifier continuous variables will be added to the new privacy budget."
            mess_text = "<span style='color: #800080; font-weight:bold; font-size: 10pt;'>"+message+"</span>"
            mess_html = widgets.HTML(value=mess_text)

            # Choose the privacy budget (epsilon) to train the synthesizer
            synt_ep_label = widgets.Label(value=r'Choose Privacy Budget ($\epsilon$) for training DP synthesizer:')
            synt_ep_boundedFloat = widgets.BoundedFloatText(value=1.0, min=0.001, max=1000.0, step=0.001, disabled=False)
            synt_ep_button = widgets.Button(description='Submit', disabled=False,
                                button_style='',
                                tooltip='submit',
                                icon='check')
            synt_hbox_ep = widgets.HBox([synt_ep_label, synt_ep_boundedFloat, synt_ep_button], layout=widgets.Layout(width='100%'))        
            
            display(synt_hbox_ep)
            synt_ep_button.on_click(on_value_change)        
                   
# Define a function that will be called when the value changes
def on_value_change(change):
    output3.clear_output()
    with output3:
        if synth == "DP: PATE-CTGAN":
            # PATE-CTGAN needs one more setting before training
            global pate_sample_size
            pate_sample_size = widgets.BoundedIntText(value=1000, min=50, max=10**7)
            pate_button = widgets.Button(description='Submit', tooltip='submit', icon='check')
            display(widgets.HBox([widgets.Label(value='Choose sample size per teacher:'),
                                  pate_sample_size, pate_button]))
            pate_button.on_click(on_pate_change)
            return

        ep = print_dp_training_info()
        fit_dp_synthesizer(ep) 


def on_pate_change(change):
    output3.clear_output()
    with output3:
        ep = print_dp_training_info() 
        teacher_sample_size = pate_sample_size.value
        print("Sample size per teacher = ", teacher_sample_size)
        print("Number of teachers = ", int(len(train_real_df) / teacher_sample_size) + 1)
        fit_dp_synthesizer(ep, sample_per_teacher=teacher_sample_size)
        
# --------- Evaluation ------------------- 
# 4. Synthetic Data Quality Evaluation.

output4 = widgets.Output() # Main and Fidelity Report 
output5 = widgets.Output() # Privacy Report
output6 = widgets.Output() # Utility Report 
output7 = widgets.Output() # SHAP values Analysis
output8 = widgets.Output() # Saving Fidelity Report  
output9 = widgets.Output() # Saving Privacy Report 
output10 = widgets.Output() # Saving Utility Report 

def add_diagnostic(output3):
    output4.clear_output()
    with output3:
        step5 = "4. Synthetic Data Quality Evaluation"
        step5_text = "<span style='color: black; font-weight:bold; font-size: 13pt;'>"+step5+"</span>"
        step5_html = widgets.HTML(value=step5_text)

        eval_radio_button = widgets.RadioButtons(
                options=['Fidelity', 'Privacy', 'Utility'],
                value=None,
                description= 'Report:',
                disabled=False)
        print()
        display(step5_html, eval_radio_button, output4)
        eval_radio_button.observe(on_eval_radio_change, 'value')

def calculate_ticks(min_value, max_value, num_ticks=5):
    """Return about num_ticks+1 evenly spaced, rounded tick values between min and max."""
    min_value, max_value = float(min_value), float(max_value)
    if not (np.isfinite(min_value) and np.isfinite(max_value)):
        return []
    if max_value <= min_value: # constant or collapsed range
        return [min_value]

    ticks = np.linspace(min_value, max_value, num_ticks + 1)
    step = (max_value - min_value) / num_ticks
    decimals = max(0, int(-np.floor(np.log10(step)))) # enough decimals to separate the ticks
    return np.unique(np.round(ticks, decimals)).tolist()

DLA_NOTE = ("Precision and recall describe how well each classifier identifies real records.<br> "
    "AUC values near 0.5 indicate that real and synthetic records are hard to tell apart.")

def on_eval_radio_change(value):
    output4.clear_output()
    output5.clear_output()
    output6.clear_output()
    output7.clear_output()
    output8.clear_output()
    output9.clear_output()
    output10.clear_output()
    with output4:
         global synthesizer
         num_cont = len(cont_var)
         num_cat = len(cat_var)+len(ord_var)
         if synth == 'GaussianCopula':
             synthesizer = synth             
         elif synth in ("DP: AIM", "DP: PATE-CTGAN"):
             synthesizer = f"{synth}, privacy budget used (ε = {total_ep}, δ = {total_delta:.0e})"             
         elif synth == "DP: MWEM":
             synthesizer = f"{synth}, privacy budget used (ε = {total_ep}, δ = 0)"            
         else:
           synthesizer = synth
                      
         # --- Prepare data for multivariate assessment ---
         ord_cols = [v[0] for v in ord_var]
         other_cols_eval = [v[0] for v in other_var]
         ord_specs = [(name, dom, metadata_obj_dict["columns"][name].get("order_by", "alphabetical"))
                      for name, dom in ord_var]

         # Drop datetime/id columns; map ordinals to ranks (same mapping in all datasets)
         train_ord, test_ord, synth_ord = util.encode_ordinals(
             [train_real_df.drop(columns=other_cols_eval, errors='ignore'),
              test_real_df.drop(columns=other_cols_eval, errors='ignore'),
              synthetic_data.drop(columns=other_cols_eval, errors='ignore')],
             ord_specs)
         train_df, test_df, synth_df = train_ord.copy(), test_ord.copy(), synth_ord.copy()

         # Scale continuous and ordinal columns (fit on Real (Train) only)
         scale_cols = [v[0] for v in cont_var] + ord_cols
         if scale_cols:
             for df in (train_df, test_df, synth_df):
                 df[scale_cols] = df[scale_cols].astype(float)
             scaler = StandardScaler().fit(train_df[scale_cols])
             for df in (train_df, test_df, synth_df):
                 df[scale_cols] = scaler.transform(df[scale_cols])

         # One-hot encode nominal columns with normalized labels
         if len(cat_var) != 0:
             cat_cols = [v[0] for v in cat_var]
             for df in (train_df, test_df, synth_df):
                 for c in cat_cols:
                     df[c] = df[c].map(util.norm_label)
             train_df = pd.get_dummies(train_df, columns=cat_cols, drop_first=False, dtype=int)
             test_df = pd.get_dummies(test_df, columns=cat_cols, drop_first=False, dtype=int)
             synth_df = pd.get_dummies(synth_df, columns=cat_cols, drop_first=False, dtype=int)

             columns_names = train_df.columns.union(test_df.columns, sort=False)
             train_df = train_df.reindex(columns=columns_names, fill_value=0)
             test_df = test_df.reindex(columns=columns_names, fill_value=0)
             synth_df = synth_df.reindex(columns=columns_names, fill_value=0)
                             
               
         if value['new'] == 'Fidelity':
             
             global fidelity_dir, uni_cont_figure_filename, uni_cont_metrics_tab_filename, uni_cat_figure_filename, uni_cat_metrics_tab_filename,\
                    bi_combined_spearman_figs, frobenius_norm, bi_combined_cramer_figs, bi_contingency_simil_file, multi_combined_prop_distr_figs, \
                    pMSE, pAUC, multi_data_labeling_table_file, multi_combined_pca_tsne_figs, multi_umap_fig
             
             fidelity_message = "Synthetic Data Fidelity Report"
             fidelity_mess_text = ("<h1 style='color: #008080; font-weight:bold;'>" + fidelity_message + "</h1>"
                                   "<span style='color: black; font-weight:bold; font-size: 16pt'> Synthesizer: </span>"
                                   f"<span style='color: black; font-size: 14pt;'>{synthesizer}</span><br>"
                                   f"<span style='font-size: 11pt;'>{seed_info()}</span>")            
             fidelity_mess_html = widgets.HTML(value=fidelity_mess_text, layout=widgets.Layout(margin='0px 0px 20px 0px'))
             
             univar_mess = "<span style='color: darkblue; font-weight:bold; font-size: 18pt'> Univariate Resemblance </span>"
             univar_mess_html = widgets.HTML(value=univar_mess, layout=widgets.Layout(margin='0px 0px 10px 0px'))
             
             display(fidelity_mess_html, univar_mess_html)
             
             # Create a directory for the fidelity report files
             fidelity_dir = res_dir+"fidelity/"             
             if os.path.exists(fidelity_dir) == False:
                 os.makedirs(fidelity_dir)  

             uni_cont_figure_filename = ""
             uni_cont_metrics_tab_filename = ""
             uni_cat_figure_filename = "" 
             uni_cat_metrics_tab_filename = ""
             bi_combined_spearman_figs = ""
             frobenius_norm = ""
             bi_combined_cramer_figs = ""
             bi_contingency_simil_file = ""
             multi_combined_prop_distr_figs = ""
             pMSE = ""
             pAUC = ""
             multi_data_labeling_table_file = ""
             multi_combined_pca_tsne_figs = ""
             multi_umap_fig = ""
                              
             # ---- Univariate Analysis -----
             
             if num_cont != 0: 
                 figures = []                
                 tab_header = ['<b>Variables</b>','<b>Total Variation</b>','<b>Hellinger</b>', '<b>Jensen-Shannon</b>']
                 cont_variables = []
                 cont_tvd_col = []
                 cont_hellin_col = []                 
                 cont_jens_col = [] 
                 
                 for c in cont_var:  
                     
                    # Create indiviual plots for Real (Train) vs Synthetic distribution of each continuous variable 
                    real = train_real_df[c[0]]                    
                    synt = synthetic_data[c[0]]
                    real_min, real_max = float(np.min(real)), float(np.max(real))
                    hist_data = [real, synt] 
                    group_labels = ['Real (Train)', 'Synthetic ']
                    colors = ['#333F44', '#37AA9C']  
                    try:
                        fig_ff = ff.create_distplot(hist_data, group_labels, colors=colors, show_rug=False, show_hist=False, curve_type='kde', histnorm='probability density')
                        data = fig_ff['data']                                      
                        fig_ff.update_layout(autosize=False, minreducedwidth=200, minreducedheight=200, width=400, height=350,
                        margin=dict(l=20, r=20, t=20, b=20, pad=2), 
                        xaxis=dict(
                        title=c[0],
                        tickvals = calculate_ticks(real_min, real_max, 6),    
                        range=[real_min, real_max], 
                        showgrid=True, 
                        zeroline=True,
                        showline=True,
                        mirror=True, 
                        gridcolor='lightgrey', 
                        gridwidth=1,
                        ),)                  
                      
                        fig = go.FigureWidget(data=fig_ff.data, layout=fig_ff.layout)
                        figures.append(fig)
                         
                        # Computing the distance metrics for each continuous variables                  
                        r_kde = stats.gaussian_kde(real)                    
                        s_kde = stats.gaussian_kde(synt)
                                                
                        # Evaluate the PDFs at x values 
                        min_val = min(min(real), min(synt))
                        max_val = max(max(real), max(synt))
                        x = np.linspace(min_val, max_val, 1000) 
                        r_pdf = r_kde.pdf(x)                                    
                        s_pdf = s_kde.pdf(x)  
                       
                        tvd_cont = metrics.tvd_continuous(r_pdf, s_pdf, x)
                        h_distance = metrics.hellinger_distance(r_pdf, s_pdf)   
                        js_distance = metrics.jensen_shannon_distance(r_pdf, s_pdf) 
                       
                        cont_variables.append(c[0]) 
                        cont_tvd_col.append(np.round(tvd_cont, decimals=3))
                        cont_hellin_col.append(np.round(h_distance, decimals=3))                       
                        cont_jens_col.append(np.round(js_distance, decimals=3))
                        
                    except np.linalg.LinAlgError as err:                       
                         print("The univariate analysis could not be computed because of the following error:", err, c[0])
                         print("Please, repeat to Step 2. and then regenerate the synthetic data.")
                         return
                    
                 cont_var_message = "Continuous Variables"
                 cont_mess_text = "<span style='color: black; font-weight:bold; font-size: 14pt'>"+cont_var_message+"</span>"
                 cont_mess_html = widgets.HTML(value=cont_mess_text)                 
                 display(cont_mess_html)
    
                 # Merging and saving continuous distributions plots for Fidelity Report                 
                 final_fig = util.merge_figurewidgets(figures)
                 uni_cont_figure_filename = "Uni_cont_distributions_figure.html"                 
                 final_fig.write_html(fidelity_dir+uni_cont_figure_filename, include_plotlyjs="cdn", full_html=False)              
                 
                 # Plotting Real (Train) vs Synthetic distributions
                 fig_layout = widgets.Layout(display='order', flex_direction='row', flex_wrap='wrap', align_content='space-between', width='auto')
                 num_figures = len(figures)
                 num_rows= num_figures // 3
                 remainder = num_figures % 3
                 indx = 0
                 figures_3_items = []
                 for i in range(num_rows):
                     figures_3_items.append(figures[indx:indx+3])
                     indx += 3 
                 if remainder:
                     figures_3_items.append(figures[indx:indx+remainder])
                 
                 fig_subplots =  [widgets.HBox(f, layout=fig_layout) for f in figures_3_items]            
                 bounds_fig = widgets.GridBox(fig_subplots, layout=widgets.Layout(height= 'auto', width='auto', grid_template_columns= 'auto', grid_template_rows= 'auto'))              
                
                 # Distance Metric Table             
                 cont_variables.append('<b>Average Distance</b>')
                 cont_tvd_col.append('<b>'+str(np.round(np.mean(cont_tvd_col), decimals=3))+'</b>')
                 cont_hellin_col.append('<b>'+str(np.round(np.mean(cont_hellin_col), decimals=3))+'</b>')
                 cont_jens_col.append('<b>'+str(np.round(np.mean(cont_jens_col), decimals=3))+'</b>') 

                 
                 dist_metric = [cont_variables, cont_tvd_col, cont_hellin_col, cont_jens_col]                 
                 tab = go.FigureWidget(data=[go.Table(header=dict(values= tab_header), cells=dict(values=dist_metric, height=25, align='center'))])
                 
                 if len(cont_variables) <= 5:
                     tab.update_layout(height=200, width=900, margin=dict(l=0, r=0, t=20, b=0),  
                                       title=dict(text="Distance metrics between synthetic and real variable distributions", x=0, xanchor="left", y=0.99, font=dict(size=14)))
                 else:               
                     tab.update_layout(width=900, margin=dict(l=0, r=0, t=20, b=0),
                                      title=dict(text="Distance metrics between synthetic and real variable distributions", x=0, xanchor="left", y=0.99, font=dict(size=14))) 

                 uni_cont_metrics_tab_filename = "Uni_cont_metrics_table.html"                 
                 tab.write_html(fidelity_dir+uni_cont_metrics_tab_filename)
                 
                 bound_tab = widgets.GridBox(children=[tab], layout=widgets.Layout(
                             grid_template_columns= 'auto',
                             grid_template_rows= 'auto',                                 
                             align_item='top',
                             height= 'auto',
                             width='1200px',
                             margin='20px 0 0 0')) 
                 
                 display(bounds_fig, bound_tab) 
                                 
             if num_cat != 0:
                               
                 cat_tab_header = ['<b>Variables</b>','<b>Total Variation</b>','<b>Hellinger</b>','<b>Jensen-Shannon</b>']
                 cat_ord_var = cat_var + ord_var                 
                 cat_figures = []
                 cat_variables = []
                 cat_tvd_col = []
                 cat_hellin_col = []
                 cat_jens_col = [] 
                 
                 for cat in cat_ord_var:  
                     
                     r_lab = train_real_df[cat[0]].map(util.norm_label)
                     s_lab = synthetic_data[cat[0]].map(util.norm_label)
                     order = util.ordinal_rank_map(pd.concat([r_lab, s_lab]), "numerical_value")
                     categories = sorted(order, key=order.get)
                     real = r_lab.value_counts().reindex(categories, fill_value=0)
                     synt = s_lab.value_counts().reindex(categories, fill_value=0)
                     
                     fig = go.Figure(data=[
                        go.Bar(
                            name='Real (Train)',
                            x=real.index, 
                            y=real.values,  
                            marker_color= '#333F44'),
                        go.Bar(
                            name='Synthetic',
                            x=synt.index,  # Categories on the x-axis
                            y=synt.values,  # Counts on the y-axis
                            marker_color='#37AA9C'  # Bar color for Category2
                        )])
                    
                     fig.update_layout(
                        barmode='group',                         
                        xaxis_title=cat[0],
                        #yaxis_title='Count',
                        template='plotly_white', autosize=False, width=400, height=350)
                     
                     fig = go.FigureWidget(data=fig.data, layout=fig.layout) 
                     cat_figures.append(fig)
                     
                     #Computing the distance metrics for the categorical variables 
                     r_prob = real/sum(real)
                     s_prob = synt/sum(synt) 
                    

                     cat_tvd = metrics.tvd_discrete(r_prob, s_prob)
                     cat_h_distance = metrics.hellinger_distance(r_prob, s_prob)
                     cat_js_distance = metrics.jensen_shannon_distance(r_prob, s_prob)
                                         
                     cat_variables.append(cat[0]) 
                     cat_tvd_col.append(np.round(cat_tvd, decimals=3))
                     cat_hellin_col.append(np.round(cat_h_distance, decimals=3)) 
                     cat_jens_col.append(np.round(cat_js_distance, decimals=3)) 
                                                                                       
                 cat_var_message = "Categorical Variables" # nominal and ordinal variables
                 cat_mess_text = "<span style='color: black; font-weight:bold; font-size: 14pt'>"+cat_var_message+"</span>"
                 cat_mess_html = widgets.HTML(value=cat_mess_text, layout=widgets.Layout(margin='0 0 0 0'))
                 display(cat_mess_html)

                 # Merging and saving categoricals bar plots for Fidelity Report                 
                 cat_final_fig = util.merge_figurewidgets(cat_figures)
                 uni_cat_figure_filename = "Uni_cat_distributions_figure.html"                 
                 cat_final_fig.write_html(fidelity_dir+uni_cat_figure_filename, include_plotlyjs="cdn", full_html=False) 
                 
                 # Plotting the distribution of Real (Train) vs Synthetic categorical variables 
                 cat_fig_layout = widgets.Layout(display='order', flex_direction='row', flex_wrap='wrap', align_content='space-between', width='auto')
                 num_cat_figures = len(cat_figures)
                 num_cat_rows= num_cat_figures // 3
                 cat_remainder = num_cat_figures % 3
                 cat_indx = 0
                 cat_figures_3_items = []
                 for i in range(num_cat_rows):
                     cat_figures_3_items.append(cat_figures[cat_indx:cat_indx+3])
                     cat_indx += 3 
                 if cat_remainder:
                     cat_figures_3_items.append(cat_figures[cat_indx:cat_indx+cat_remainder])
                 
                 cat_fig_subplots =  [widgets.HBox(f, layout=cat_fig_layout) for f in cat_figures_3_items]            
                 cat_bounds_fig = widgets.GridBox(cat_fig_subplots, layout=widgets.Layout(height= 'auto', width='auto', grid_template_columns= 'auto', grid_template_rows= 'auto'))              

                 # Distance Metric Table
                 cat_variables.append('<b>Average Distance</b>')
                 cat_tvd_col.append('<b>'+str(np.round(np.mean(cat_tvd_col), decimals=3))+'</b>')
                 cat_hellin_col.append('<b>'+str(np.round(np.mean(cat_hellin_col), decimals=3))+'</b>')
                 cat_jens_col.append('<b>'+str(np.round(np.mean(cat_jens_col), decimals=3))+'</b>') 

                 cat_dist_metric = [cat_variables, cat_tvd_col, cat_hellin_col, cat_jens_col]
                 
                 cat_tab = go.FigureWidget(data=[go.Table(header=dict(values=cat_tab_header), cells=dict(values=cat_dist_metric, height=25, align='center'))])  
                 if len(cat_variables) <= 5:
                     cat_tab.update_layout(height=200, width=900, margin=dict(l=0, r=0, t=20, b=0),  
                                       title=dict(text="Distance metrics between synthetic and real distributions", x=0, xanchor="left", y=0.99, font=dict(size=14)))                     
                 else:               
                     cat_tab.update_layout(width=900, margin=dict(l=0, r=0, t=20, b=0),
                                      title=dict(text="Distance metrics between synthetic and real distributions", x=0, xanchor="left", y=0.99, font=dict(size=14)))

                 uni_cat_metrics_tab_filename = "Uni_cat_metrics_table.html"                 
                 cat_tab.write_html(fidelity_dir+uni_cat_metrics_tab_filename)
                
                 cat_bound_tab = widgets.GridBox(children=[cat_tab], layout=widgets.Layout(
                              grid_template_columns= 'auto',
                              grid_template_rows= 'auto',                                 
                              align_item='top',
                              height= 'auto',
                              width='1200px',
                              margin='20px 0 0 0'))
                 
                 display(cat_bounds_fig, cat_bound_tab)
                 
                     
             #------ Bivariate Analysis --------
             
             full_colorscale = px.colors.diverging.BrBG 
             bg_half = full_colorscale[len(full_colorscale)//2:] # Custom colorscale for heatmaps                  
                                              
             bivar_mess = "<span style='color: darkblue; font-weight:bold; font-size: 18pt;'> Bivariate Resemblance </span>"
             bivar_mess_html = widgets.HTML(value=bivar_mess, layout=widgets.Layout(margin='0px 0px 10px 0px'))             
             display(bivar_mess_html) 
    
             # Spearman's Correlation for Continuous and Ordinal variables
             cont_ord_var = cont_var + ord_var 
             nom_ord_var = cat_var + ord_var
            
             if len(cont_ord_var) >= 2:
                 pair_corr_message = "Continuous and Ordinal Variables"
                 pair_corr_text = "<span style='color: black; font-weight:bold; font-size: 14pt;'>"+pair_corr_message+"</span>"
                 pair_corr_mess_html = widgets.HTML(value=pair_corr_text, layout=widgets.Layout(margin='0 0 15px 0')) 
                 display(pair_corr_mess_html)
                 
                 var_names = [var[0] for var in cont_ord_var]
                 
                 # Create a Figure Widget for Real (Train) data heatmap 
                 real_data = train_ord[var_names]                
                 real_corr = metrics.corr_matrix(real_data, method='spearman') 
                 real_mask = np.triu(np.ones_like(real_corr, dtype=bool), k=1)                     
                 
                 width_v, height_v = 700, 500                     
                 if len(cont_ord_var) < 5: 
                     width_v, height_v = 500, 300
                 elif len(cont_ord_var) > 10:
                     width_v, height_v = 850, 550
                 
                 real_layout = go.Layout(title=dict(text="Real (Train) Spearman's Correlation Matrix", font=dict(size=14, color='black')), autosize=True, margin=dict(l=50, r=50, t=50, b=50),
                               xaxis_showgrid=False, yaxis_showgrid=False, xaxis_zeroline=False, yaxis_zeroline=False, yaxis_autorange='reversed', template='plotly_white', width=width_v, height=height_v)                     
                 real_fig = go.Figure(data=(go.Heatmap(z=real_corr.mask(real_mask), x=real_corr.columns, y=real_corr.columns, colorscale= bg_half, zmin=-1, zmax=1, 
                                           text=real_corr.values.round(2), texttemplate="%{text}", textfont={"size":14})), layout=real_layout)                     
                 real_figw = go.FigureWidget(data=real_fig.data, layout=real_fig.layout)
                                   
                 # Create a Figure Widget for Synth data heatmap
                 synth_data = synth_ord[var_names] #synthetic_data[var_names]
                 synth_corr = metrics.corr_matrix(synth_data, method='spearman')
                 synth_mask = np.triu(np.ones_like(synth_corr, dtype=bool), k=1)                    
                 
                 synth_layout = go.Layout(title=dict(text="Synthetic Spearman's Correlation Matrix", font=dict(size=14, color='black')), autosize=True, margin=dict(l=50, r=50, t=50, b=50),
                               xaxis_showgrid=False, yaxis_showgrid=False, xaxis_zeroline=False, yaxis_zeroline=False, yaxis_autorange='reversed', template='plotly_white', width=width_v, height=height_v) 
                 synth_fig = go.Figure(data=(go.Heatmap(z=synth_corr.mask(synth_mask), x=synth_corr.columns, y=synth_corr.columns, colorscale= bg_half, zmin=-1, zmax=1, 
                                           text=synth_corr.values.round(2), texttemplate="%{text}", textfont={"size":14})), layout=synth_layout)
                 synth_figw = go.FigureWidget(data=synth_fig.data, layout=synth_fig.layout) 
                 
                 # Save Spearman's heatmaps side by side for the report 
                 bi_real_spearman_html = pio.to_html(real_figw, full_html=False, include_plotlyjs='cdn')
                 bi_synth_spearman_html = pio.to_html(synth_figw, full_html=False, include_plotlyjs=False)
                 bi_combined_spearman_html = f"""
                                <div style="width: 794px; display: flex; gap: 10px;">
                                    {bi_real_spearman_html}
                                    {bi_synth_spearman_html}
                                </div>
                                """
                 bi_combined_spearman_figs = "Bi_combined_spearman_figures.html"
                 f = open(fidelity_dir+bi_combined_spearman_figs, "w", encoding="utf-8")
                 f.write(bi_combined_spearman_html)
                 f.close()

                 heat_subfig_layout = widgets.Layout(display='order', flex_direction='row', flex_wrap='wrap', align_content='space-between', width='auto')                 
                 heat_subfig = widgets.HBox([real_figw, synth_figw], layout=heat_subfig_layout)
                
                 heat_bound_fig = widgets.GridBox(children=[heat_subfig], layout=widgets.Layout(
                              grid_template_columns= 'auto',
                              grid_template_rows= 'auto',  
                              margin='0 0 5px 0',
                              align_item='top',
                              height= 'auto',
                              width='auto'))
                 
                 # Compute Frobenius Norm
                 corr_diff = np.tril(real_corr) - np.tril(synth_corr)
                 frobenius_norm = np.linalg.norm(corr_diff, 'fro').round(3)                        
                
                 pair_corr_text = ("<span style='color:#0f736b; font-weight:700; font-size: 14pt'>Real (Train) vs Synthetic: </span>"
                                   "<span style='color:black; font-size: 14pt'> Pairwise correlation</span>")
                 pair_corr_html = widgets.HTML(value=pair_corr_text) 
             
                 forb_norm_text = (
                                 "<div style='color: black; font-size: 12pt; margin-top: 0; margin-bottom:40px; padding-left: 20px;'>"
                                 f"<b>Frobenius Norm</b> = {np.round(frobenius_norm, 3)} "
                                 "<i class='fa fa-info-circle' "
                                  "style='font-size: 12px; color:#bcc0c0; background:white; border-radius: 50%; padding: 2px; cursor: pointer;' "
                                  "title='Computed from the difference of the Spearman&#39;s matrices'></i>"                           
                                 "</div>")
                 
                 forb_mess_html = widgets.HTML(value=forb_norm_text)                                     
                 display(pair_corr_html, heat_bound_fig, forb_mess_html)                       
                 
             if len(nom_ord_var) >= 2:                      
                 cat_var_message = "Nominal and Ordinal Variables" 
                 cat_mess_text = "<span style='color: black; font-weight:bold; font-size: 14pt'>"+cat_var_message+"</span>"
                 cat_mess_html = widgets.HTML(value=cat_mess_text, layout=widgets.Layout(margin='0 0 15px 0'))
                 display(cat_mess_html)
                                 
                 var_names = [var[0] for var in nom_ord_var]
                 real_data = train_real_df[var_names]
                 synth_data = synthetic_data[var_names].copy()
                 
                 # Make synthetic categories comparable with the real ones (e.g. "1" vs 1)
                 for c in var_names:
                     if synth_data[c].dtype != real_data[c].dtype:
                         try:
                             synth_data[c] = synth_data[c].astype(real_data[c].dtype)
                         except (ValueError, TypeError):
                             synth_data[c] = synth_data[c].astype(str)
                             real_data = real_data.astype({c: str})
                 
                 # Matrices in the dataset's column order (no alphabetical re-sorting)
                 real_cramer_df = pd.DataFrame(np.nan, index=var_names, columns=var_names)
                 synth_cramer_df = pd.DataFrame(np.nan, index=var_names, columns=var_names)
                 df_similar_score = pd.DataFrame(np.nan, index=var_names, columns=var_names)

                 # Diagonal: V = 1 for a variable with itself (NaN if constant);
                 # contingency similarity = 1 - univariate TVD
                 for a in var_names:
                     real_cramer_df.loc[a, a] = 1.0 if real_data[a].nunique() > 1 else np.nan
                     synth_cramer_df.loc[a, a] = 1.0 if synth_data[a].nunique() > 1 else np.nan
                     df_similar_score.loc[a, a] = metrics.contingency_similarity(
                         real_data[a], real_data[a], synth_data[a], synth_data[a])

                 # Off-diagonal: each unordered pair once, then mirrored (both metrics are symmetric)
                 for a, b in itertools.combinations(var_names, 2):
                     r_v = metrics.cramers_v(pd.crosstab(real_data[a], real_data[b]))
                     s_v = metrics.cramers_v(pd.crosstab(synth_data[a], synth_data[b]))
                     sim = metrics.contingency_similarity(real_data[a], real_data[b],
                                                          synth_data[a], synth_data[b])
                     real_cramer_df.loc[a, b] = real_cramer_df.loc[b, a] = r_v
                     synth_cramer_df.loc[a, b] = synth_cramer_df.loc[b, a] = s_v
                     df_similar_score.loc[a, b] = df_similar_score.loc[b, a] = sim

                 # Tell the user why some cells are empty
                 constant_synth = [c for c in var_names if synth_data[c].nunique() < 2]
                 if constant_synth:
                     display(widgets.HTML(
                         "<div style='color:darkred; background-color:LightYellow; font-size:10pt;'>"
                         f"The synthetic data has only one category for: <b>{', '.join(map(str, constant_synth))}</b>. "
                         "Cramér's V is undefined for these variables (empty cells in the heatmap).</div>"))

                     
                 # Plot Cramér's V matrix for Real (Train) and Synthetic for visual comparison
                 cramer_text = ("<span style='color:#0f736b; font-weight:700; font-size: 14pt'>Real (Train) vs Synthetic: </span>"
                                "<span style='color:black; font-size: 14pt'>Pairwise association</span>")
                 
                 cramer_mess_html = widgets.HTML(value=cramer_text) 
                 display(cramer_mess_html)

                 # Real (Train) Cramer's V heatmap
                 real_mask = np.triu(np.ones_like(real_cramer_df, dtype=bool), k=1)
                                     
                 width_v, height_v = 700, 500                     
                 if len(nom_ord_var) < 7: 
                     width_v, height_v = 500, 300 
                 elif len(nom_ord_var) > 10:
                     width_v, height_v = 850, 550
             
                 real_cramer_heatmap = go.Heatmap(z=real_cramer_df.mask(real_mask), x=real_cramer_df.columns, y=real_cramer_df.columns, colorscale= bg_half, zmin=0, zmax=1, 
                                           text=real_cramer_df.values.round(2), texttemplate="%{text}", textfont={"size":14})
                 real_cramer_layout = go.Layout(title=dict(text="Real (Train) Cramer's V", font=dict(size=14, color='black')), autosize=True, margin=dict(l=50, r=50, t=50, b=50),
                                           xaxis_showgrid=False, yaxis_showgrid=False, xaxis_zeroline=False, yaxis_zeroline=False, yaxis_autorange='reversed', template='plotly_white')
                 real_cramer_fig = go.Figure(data=[real_cramer_heatmap], layout=real_cramer_layout) 
                 real_cramer_fig.update_layout(xaxis_title='', yaxis_title='', font=dict(size=12, color='black'), width=width_v, height=height_v)
                 real_cramer_figw = go.FigureWidget(data=real_cramer_fig.data, layout=real_cramer_fig.layout)
                
                 # Synthetic Cramer's V heatmap
                 synth_mask = np.triu(np.ones_like(synth_cramer_df, dtype=bool), k=1)
                                  
                 synth_cramer_heatmap = go.Heatmap(z=synth_cramer_df.mask(synth_mask), x=synth_cramer_df.columns, y=synth_cramer_df.columns, colorscale= bg_half, zmin=0, zmax=1, 
                                                   text=synth_cramer_df.values.round(2), texttemplate="%{text}", textfont={"size":14})
                 synth_cramer_layout = go.Layout(title=dict(text="Synthetic Cramer's V", font=dict(size=14, color='black')), autosize=True, margin=dict(l=50, r=50, t=50, b=50),
                                                 xaxis_showgrid=False, yaxis_showgrid=False, xaxis_zeroline=False, 
                                                 yaxis_zeroline=False, yaxis_autorange='reversed', template='plotly_white')
                 synth_cramer_fig = go.Figure(data=[synth_cramer_heatmap], layout=synth_cramer_layout) 
                 synth_cramer_fig.update_layout(xaxis_title='', yaxis_title='', font=dict(size=12, color='black'), width=width_v, height=height_v)
                 synth_cramer_figw = go.FigureWidget(data=synth_cramer_fig.data, layout=synth_cramer_fig.layout)

                
                 # Save Cramer's V heatmaps side by side for the report 
                 bi_real_cramer_html = pio.to_html(real_cramer_figw, full_html=False, include_plotlyjs='cdn')
                 bi_synth_cramer_html = pio.to_html(synth_cramer_figw, full_html=False, include_plotlyjs=False)
                 bi_combined_cramer_html = f"""
                                <div style="width: 794px; display: flex; gap: 10px;">
                                    {bi_real_cramer_html}
                                    {bi_synth_cramer_html}
                                </div>
                                """                                    
                 bi_combined_cramer_figs = "Bi_combined_cramers_v_figs.html"
                 f = open(fidelity_dir+bi_combined_cramer_figs, "w", encoding="utf-8")
                 f.write(bi_combined_cramer_html)
                 f.close()

                 cramer_subfig_layout = widgets.Layout(display='order', flex_direction='row', flex_wrap='wrap', align_content='space-between', width='auto')                 
                 cramer_subfig = widgets.HBox([real_cramer_figw, synth_cramer_figw], layout=cramer_subfig_layout)
                
                 cramer_bound_fig = widgets.GridBox(children=[cramer_subfig], layout=widgets.Layout(
                              grid_template_columns= 'auto',
                              grid_template_rows= 'auto',
                              margin = '0 0 20px 0',
                              align_item='top',
                              height= 'auto',
                              width='auto'))
                 
                 display(cramer_bound_fig)
                
                 # Plot the Two-way Contingency Similarity Score                   
                 contigency_text = ("<span style='color:#0f736b; font-weight:700; font-size: 14pt'>Real (Train) vs Synthetic:</span>"
                                    "<span style='color:black; font-size: 14pt'> Two-way contingency </span>")
                 contigency_mess_html = widgets.HTML(value=contigency_text) 
                 display(contigency_mess_html) 

                 mask = np.triu(np.ones_like(df_similar_score, dtype=bool), k=1)
                                     
                 simil_heatmap = go.Heatmap(z=df_similar_score.mask(mask), x=df_similar_score.columns, y=df_similar_score.columns, colorscale= bg_half, zmin=0, zmax=1, 
                                           text=df_similar_score.values.round(2), texttemplate="%{text}", textfont={"size":14})
                 simil_layout = go.Layout(title=dict(text="Similarity score (1= equal and 0= different)", font=dict(size=14, color='black')), autosize=True, margin=dict(l=50, r=50, t=50, b=50), xaxis_showgrid=False, yaxis_showgrid=False, xaxis_zeroline=False, yaxis_zeroline=False, 
                                          yaxis_autorange='reversed', template='plotly_white')
                 simil_fig = go.Figure(data=[simil_heatmap], layout=simil_layout) 
                 simil_fig.update_layout(font=dict(size=12, color='black'), width=width_v, height=height_v)

                 bi_contingency_simil_file = "Bi_contingency_similarity_figure.html"
                 simil_fig.write_html(fidelity_dir+bi_contingency_simil_file)
                 
                 simil_fig.update_layout(xaxis_title='', yaxis_title='', font=dict(size=12, color='black'), width=width_v, height=height_v)
                 simil_figw = go.FigureWidget(data=simil_fig.data, layout=simil_fig.layout)
                 simil_subfig = widgets.HBox([simil_figw], layout=widgets.Layout(margin='0 0 25px 0'))
                 display(simil_subfig)

             #------- Multivariate Analysis ---------
             
             multi_mess = "<span style='color: darkblue; font-weight:bold; font-size: 18pt'> Multivariate Resemblance </span>"
             multi_mess_html = widgets.HTML(value=multi_mess, layout=widgets.Layout(margin='0px 30px 15px 0px')) 
                                           
             # Propensity Score 
             prop_score_mess =  ("<span style='color:#0f736b; font-weight:700; margin-bottom: 5px; font-size: 14pt'>"
                                 "Propensity Score based on Logistic Regression</span>")
             prop_score_mess_html = widgets.HTML(value=prop_score_mess)
             display(multi_mess_html, prop_score_mess_html)
             
             # Use the scaled and encoded version of the train and synthetic datasets      
             r_df = train_df.copy() 
             s_df = synth_df.copy() 
             r_df["Label"] = np.ones(r_df.shape[0]).astype('int8')
             s_df["Label"] = np.zeros(s_df.shape[0]).astype('int8') 
                            
             # Merge Real (Train) and  Synthetic Dataframes
             merged_data = pd.concat([r_df, s_df], ignore_index=True)  
             
             y_train = merged_data["Label"].to_numpy()
             X_train = merged_data.drop(columns=["Label"])
                                                       
             # Log Regression as the classifier to compute the propensity score
             log_reg = LogisticRegression(solver='liblinear')               
                              
             try:
                 log_reg.fit(X_train, y_train)
                 prop_score = log_reg.predict_proba(X_train)[:, 1]   # P(record is Real)
                 merged_data["Prop_Score"] = prop_score
                 
                 # Mean Propensity Score 
                 mean_pscore = np.mean(prop_score)                                    
                 if mean_pscore < 0.4 or mean_pscore > 0.6:
                    print("Warning: The synthetic data may not be well aligned with the real data.")
    
                 # Propensity Score Mean Square Error (Woo et al. 2009, eq. 1)
                 c = y_train.mean() # share of Real records (the class predicted by p)
                 pMSE = np.round(np.mean((prop_score - c) ** 2), 6)
                 pAUC= np.round(roc_auc_score(y_train, prop_score), 2)          
                 
                 metrics_text =("<ul style='color: black; font-size: 12pt; margin-top: 5px; padding-left: 18px;'>"
                                f"<li>In-sample model <b>AUC</b> = {pAUC}</li>"
                                f"<li>Propensity score <b>MSE</b> = {pMSE} "
                                "<i class='fa fa-info-circle' "
                                "style='font-size: 12px; color:#bcc0c0; background:white; border-radius: 50%; padding: 2px; cursor: pointer;' "
                                "title='Estimate Propensity Score Mean Square Error using equation (1) in Mi-Ja Woo 2009'></i></li>"
                                "</ul>")                   
                
                 prop_metrics = widgets.HTML(value=metrics_text) 
                 
                 display(prop_metrics)  
                 
                 # Compare Propensity score Distribution between Real (Train) and Synthetic data
                 r_prop = merged_data[merged_data["Label"] == 1]["Prop_Score"] 
                 s_prop = merged_data[merged_data["Label"] == 0]["Prop_Score"]
                                     
                 dist_fig = go.Figure()
                 dist_fig.add_trace(go.Histogram(x=r_prop, name="Real (Train)", opacity=0.6, marker_color="#333F44", nbinsx=30))
                 dist_fig.add_trace(go.Histogram(x=s_prop, name="Synthetic", opacity=0.6, marker_color="#37AA9C", nbinsx=30))
                              
                 dist_layout = go.Layout(title=dict(text="Propensity Score Distribution", font=dict(size=14, color='black')), barmode='overlay', 
                                         width=500, height=350, margin=dict(l=50, r=50, t=50, b=50), template='plotly_white', xaxis_title="Propensity Score", yaxis_title="Count")
                 dist_figw = go.FigureWidget(data=dist_fig.data, layout=dist_layout)
    
                 #Compare Empirical CDF of Propensity Scores
                 merged_data.loc[merged_data["Label"] == 1, "DataType"] = "Real (Train)"
                 merged_data.loc[merged_data["Label"] == 0, "DataType"] = "Synthetic"            
                 ecdf_fig = px.ecdf(merged_data, x="Prop_Score", color="DataType", color_discrete_map={"Real (Train)": "#333F44", "Synthetic": "#37AA9C"})
                 ecdf_fig.update_layout(title=dict(text="Empirical CDF of Propensity Scores", font=dict(size=14, color='black')), width=500, height=350, margin=dict(l=50, r=50, t=50, b=50), 
                                        template="plotly_white", xaxis_title="Propensity Score", yaxis_title="Probability", legend_title_text=None)
                 ecdf_figw = go.FigureWidget(data=ecdf_fig.data, layout=ecdf_fig.layout)

                 # Save figures for the fidelity report
                 multi_prop_distr_html = pio.to_html(dist_figw, full_html=False, include_plotlyjs='cdn')
                 multi_prop_ecdf_html = pio.to_html(ecdf_figw, full_html=False, include_plotlyjs=False)
                 multi_combined_distr_html = f"""
                                <div style="display: flex; gap: 10px;">
                                    {multi_prop_distr_html}
                                    {multi_prop_ecdf_html}
                                </div>
                                """
                 multi_combined_prop_distr_figs = "Multi_combined_propensity_distr_figures.html"
                 f = open(fidelity_dir+multi_combined_prop_distr_figs, "w", encoding="utf-8")
                 f.write(multi_combined_distr_html)
                 f.close()
                 
                 prop_subfig = widgets.HBox([dist_figw, ecdf_figw], layout=widgets.Layout(margin='0 0 25px 0'))                            
                 display(prop_subfig)
                 
             except ValueError as e:  
                 print(f"Cannot compute the propensity score: {e}")

             # Data Labeling Analysis
             dla_mess =  ("<div style='color:#0f736b; font-weight:700; margin-top:20px; font-size: 14pt'>" 
                          "Data Labeling Analysis</div>"                             
                          "<div style ='font-size: 12pt; margin-top: 5px;'>Metrics are the mean values from 5-fold cross-validation.</div>")
             
             dla_html = widgets.HTML(value=dla_mess)
             display(dla_html)
             
             skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)                 
             classifiers = {'5-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5), 
                            'Decision Tree': DecisionTreeClassifier(random_state=SEED),
                            'Random Forest': RandomForestClassifier(random_state=SEED)}
            
             scoring = ['precision', 'recall', 'f1', 'roc_auc']
             dla_cls_metric = []
             for name, cls in classifiers.items():
                 cv_res = cross_validate(cls, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)
                 dla_cls_metric.append([name] + [f"{np.mean(cv_res['test_' + s]):.2f}" for s in scoring])
                 
             dla_header = ["<b>Classifier</b>", "<b>Precision</b>", "<b>Recall</b>", "<b>F1 score</b>", "<b>AUC</b>"]               
    
             dla_tab = go.FigureWidget(data=[go.Table(header=dict(values=dla_header, align='left'), columnwidth=[200, 150, 100, 100, 100],
                                              cells=dict(values=[[row[i] for row in dla_cls_metric] for i in range(5)], height=25, align='left'))]) 
             dla_tab.update_layout(width=650, height=110, margin=dict(l=0, r=0, t=0, b=0))  

             multi_data_labeling_table_file = "Multi_data_labeling_table.html"
             dla_tab.write_html(fidelity_dir+multi_data_labeling_table_file)
    
             dla_tab_grid = widgets.GridBox(children=[dla_tab], layout=widgets.Layout(
                                            grid_template_columns= 'auto',
                                            grid_template_rows= 'auto', 
                                            grid_gap='0 0',                                             
                                            align_item='flex-start',
                                            margin="0 0 10px 0", 
                                            height='110px')) 
             
             dla_note_html = widgets.HTML(
                 f"<div style='font-size:11pt; line-height:1.5; margin:0;'>{DLA_NOTE}</div>",
                 layout=widgets.Layout(margin='0px'))
             
             display(widgets.VBox([dla_tab_grid, dla_note_html],
                                  layout=widgets.Layout(margin='0px 0px 10px 0px'))) 
             
             # Dimensionality reduction techniques              
             dim_reduc_mess =  "<div style='color:#0f736b; font-weight:700; margin-top:20px; margin-bottom: 5px; font-size: 14pt'> Visualization with Dimensionality Reduction Techniques </div>"
             dim_reduc_mess_html = widgets.HTML(value=dim_reduc_mess)
             display(dim_reduc_mess_html)
             
             # Use the scaled and encoded version of the train and synthetic datasets
             re_df =  train_df.copy()
             sy_df = synth_df.copy()
            
             real_labels = ["Real (Train)"] * len(re_df)
             synth_labels = ["Synthetic"] * len(sy_df)
             all_labels = np.array(real_labels + synth_labels)
             
             # PCA and t-SNE for continuous variables
             
             if len(cont_var) >= 2:
                 
                 display(cont_mess_html) # Display "Continuous Variables"
                 cont_cols = list(np.array(cont_var, dtype='object')[:, 0])

                 real_cont = re_df[cont_cols].to_numpy(dtype=float)
                 synth_cont = sy_df[cont_cols].to_numpy(dtype=float)
                 n_real = real_cont.shape[0]
                 
                 # --- PCA: fit on Real (Train) only, then project both datasets onto the same axes ---
                 pca = PCA(n_components=2, random_state=SEED)
                 pca.fit(real_cont)
                 real_pca = pca.transform(real_cont)
                 synth_pca = pca.transform(synth_cont)
                 all_pca = np.vstack((real_pca, synth_pca))

                 var_pc1, var_pc2 = pca.explained_variance_ratio_ * 100

                 pca_fig = px.scatter(x=all_pca[:, 0], y=all_pca[:, 1], color=all_labels,
                                      color_discrete_map={"Real (Train)": "#333F44", "Synthetic": "#37AA9C"},
                                      opacity=0.8)
                 pca_fig.update_layout(title=dict(text="PCA (Principal Component Analysis)", font=dict(size=14, color='black')),
                                       width=600, height=350, margin=dict(l=50, r=50, t=50, b=50),
                                       template="plotly_white",
                                       xaxis_title=f"PC1 ({var_pc1:.1f}% of variance)",
                                       yaxis_title=f"PC2 ({var_pc2:.1f}% of variance)",
                                       legend_title_text=None)
                 pca_figw = go.FigureWidget(data=pca_fig.data, layout=pca_fig.layout)

                                  
                 # --- t-SNE: run once on Real (Train) + Synthetic combined, then split ---
                 combined_cont = np.vstack((real_cont, synth_cont))
                 perplexity = min(30.0, (combined_cont.shape[0] - 1) / 3)  # must be < number of samples
                 tsne_model = TSNE(n_components=2, perplexity=perplexity, init="pca", random_state=SEED)
                 all_tsne = tsne_model.fit_transform(combined_cont)
                 real_tsne, synth_tsne = all_tsne[:n_real], all_tsne[n_real:]

                 tsne_fig = px.scatter(x=all_tsne[:, 0], y=all_tsne[:, 1], color=all_labels,
                                       color_discrete_map={"Real (Train)": "#333F44", "Synthetic": "#37AA9C"},
                                       opacity=0.8)
                 tsne_fig.update_layout(title=dict(text="t-SNE (t-distributed Stochastic Neighbor Embedding)", font=dict(size=14, color='black')),
                                        width=600, height=350, margin=dict(l=50, r=50, t=50, b=50),
                                        template="plotly_white", xaxis_title="First t-SNE", yaxis_title="Second t-SNE",
                                        legend_title_text=None)
                 tsne_figw = go.FigureWidget(data=tsne_fig.data, layout=tsne_fig.layout)

                 # Save PCA and t-SNE plots side by side for the report
                 multi_pca_html = pio.to_html(pca_figw, full_html=False, include_plotlyjs='cdn')
                 multi_tsne_html = pio.to_html(tsne_figw, full_html=False, include_plotlyjs=False)
                 multi_combined_pca_tsne_html = f"""
                                <div style="display: flex; gap: 10px;">
                                    {multi_pca_html}
                                    {multi_tsne_html}
                                </div>
                                """
                 multi_combined_pca_tsne_figs = "Multi_combined_pca_tsne_figs.html"
                 with open(fidelity_dir + multi_combined_pca_tsne_figs, "w", encoding="utf-8") as f:
                     f.write(multi_combined_pca_tsne_html)

                 pca_tsne_subfig = widgets.HBox([pca_figw, tsne_figw])
                 display(pca_tsne_subfig)
                 
             # UMAP
             all_var_message = "All Variables"
             all_var_text = "<div style='color: black; font-weight:bold; margin-top:15px; font-size: 14pt'>"+all_var_message+"</div>"
             all_var_mess_html = widgets.HTML(value=all_var_text) 
             display(all_var_mess_html) 

             try:
                 umap_model = umap.UMAP(n_components=2, random_state=SEED)
                 real_umap = umap_model.fit_transform(re_df)
                 synth_umap = umap_model.transform(sy_df)
                 all_umap = np.vstack((real_umap, synth_umap))                   
    
                 umap_fig = px.scatter(x=all_umap[:, 0], y=all_umap[:, 1], color=all_labels, color_discrete_map={"Real (Train)": "#333F44", "Synthetic": "#37AA9C"}, opacity=0.8)
                 umap_fig.update_layout(title=dict(text="UMAP (Uniform Manifold Approximation and Projection)", font=dict(size=14, color='black')), width=600, height=350, margin=dict(l=50, r=50, t=50, b=50), 
                                            template="plotly_white", xaxis_title="First UMAP", yaxis_title="Second UMAP", legend_title_text=None)
                 
                 umap_figw = go.FigureWidget(data=umap_fig.data, layout=umap_fig.layout)

                 multi_umap_fig = "Multi_umap_figure.html"
                 umap_figw.write_html(fidelity_dir+multi_umap_fig)
                 
                 umap_subfig = widgets.HBox([umap_figw])                            
                 display(umap_subfig)
                 
             except ValueError as e:
                 print("ValueError: Cannot compute UMAP — synthetic data may contain NaNs or invalid values.")   
                 if re_df.shape != sy_df.shape:                       
                    print("Shape mismatch after One-Hot Encoding.")
                    print("Real (Train):", re_df.shape, "| Synthetic:", sy_df.shape)
                    # Symmetric difference (columns that are not shared)
                    diff = set(re_df.columns).symmetric_difference(set(sy_df.columns))
                    print(diff)                         
                 else:
                    print(e)
                         
             # Button for saving the fidelity report
             print()
             fidelity_report_btn = widgets.Button(
                                                 description='Save Fidelity Report',
                                                 disabled=False,
                                                 button_style='', 
                                                 tooltip='Save Fidelity Report',
                                                 icon='check',
                                                 layout=widgets.Layout(width='200px'),
                                                 style={'font_size': '14px'})
             fidelity_report_btn.on_click(fidelity_btn_clicked)
             display(fidelity_report_btn, output8)  

         elif value['new'] == 'Privacy': 
             
                global min_dcr_float, tt_distances, tt_closest_idx, \
                th_distances, th_closest_idx, ts_distances,\
                ts_closest_idx, hs_distances, hs_closest_idx, \
                train_priv, test_priv, synth_priv, gower_ref
             
                privacy_message = "Synthetic Data Privacy Report"
                privacy_mess_text = ("<h1 style='color: #008080; font-weight:bold;'>" + privacy_message + "</h1>"
                                   "<span style='color: black; font-weight:bold; font-size: 16pt'> Synthesizer: </span>"
                                   f"<span style='color: black; font-size: 14pt;'>{synthesizer}</span><br>"
                                   f"<span style='font-size: 11pt;'>{seed_info()}</span>") 
                
                privacy_mess_html = widgets.HTML(value=privacy_mess_text)
             
                dcr_title = "<div style='color: darkblue; font-weight:bold; font-size: 16pt; margin-top: 10px; margin-bottom: 10px;'> Distance to Closest Record (DCR)</div>"
                dcr_title_html = widgets.HTML(value=dcr_title)
                dcr_text_html = widgets.HTML(value="""<div style="font-size: 11pt; line-height: 1.5;">
                                    DCR measures the similarity between records in two datasets, for example, between Train (Real) and Synthetic records.<br>
                                    A low DCR alone <b>does not imply identity disclosure</b>.<br> 
                                    It should be interpreted as a proximity indicator, not evidence of a privacy breach.<br></div>""")
             
                display(privacy_mess_html, dcr_title_html, dcr_text_html)
                print()
                               
                # Compare only the modelled columns (datetime/id excluded)
                priv_cols = [c for c in train_real_df.columns if c not in other_cols]
                train_priv = train_real_df[priv_cols]
                test_priv = test_real_df[priv_cols]
                synth_priv = synthetic_data[priv_cols]

                # One Gower scale for every comparison: ranges from the Real (Train) set,
                # categorical columns from the metadata
                gower_ref = metrics.fit_gower_params(train_priv, [v[0] for v in cat_var + ord_var])
             
                # Compute Gower's distance matrix between among the data points in Real Train (TTD)
                ttd_spinner = widgets.IntProgress(value=0, min=0, max=0, description="Computing Train-to-Train DCR...", 
                                                  style={'description_width': '250px'}, layout=widgets.Layout(width='500px'))
                display(ttd_spinner)                
                tt_distances, tt_closest_idx = metrics.gower_distances(train_priv, params=gower_ref)
                ttd_spinner.close()

                # Compute Gower's distance matrix between Train to Holdout (THD)
                thd_spinner = widgets.IntProgress(value=0, min=0, max=0, description='Computing Train-to-Holdout DCR...', 
                                                  style={'description_width': '250px'}, layout=widgets.Layout(width='500px'))
                display(thd_spinner)
                th_distances, th_closest_idx = metrics.gower_distances(train_priv, test_priv, params=gower_ref)
                thd_spinner.close() 
            
                # Compute Gower's distance matrix between  Real Train to Synthetic (TSD)
                tsd_spinner = widgets.IntProgress(value=0, min=0, max=0, description='Computing Train-to-Synthetic DCR...', 
                                                  style={'description_width': '250px'}, layout=widgets.Layout(width='500px'))
                display(tsd_spinner)
                ts_distances, ts_closest_idx = metrics.gower_distances(train_priv, synth_priv, params=gower_ref)               
                tsd_spinner.close()
             
                # Compute Gower's distance matrix between Holdout to Synthetic (HSD)
                hsd_spinner = widgets.IntProgress(value=0, min=0, max=0, description='Computing Holdout-to-Synthetic DCR...', 
                                                  style={'description_width': '250px'}, layout=widgets.Layout(width='500px'))
                display(hsd_spinner)
                hs_distances, hs_closest_idx = metrics.gower_distances(test_priv, synth_priv, params=gower_ref)
                hsd_spinner.close()
             
                # Record Matching 
                record_match_mess =  "<span style='color:#0f736b; font-weight:700; font-size: 14pt'> Record Matching </span>"
                record_match_mess_html = widgets.HTML(value=record_match_mess)
                display(record_match_mess_html)
             
                # Choose the distance min_dcr
                min_dcr_label = widgets.Label(value="Minimum DCR:")
                min_dcr_float = widgets.BoundedFloatText(value=0.0, min=0.0, max=1.0, step=0.01, disabled=False)
                min_dcr_button = widgets.Button(description='Submit', disabled=False, button_style='', tooltip='submit', icon='check')
                min_dcr_hbox = widgets.HBox([min_dcr_label, min_dcr_float, min_dcr_button], layout=widgets.Layout(width='80%', margin='0px 0px 25px 0px')) 
                display(min_dcr_hbox, output5)
                min_dcr_button.on_click(min_dcr_value_change)
                 
         elif value['new'] == 'Utility':
             global cont_names, ord_names, cat_names, target_widget, estimator_widget, utility_dir, summary_stats_filename

             # Compare only the modelled columns (datetime/id excluded)
             util_cols = [c for c in train_real_df.columns if c not in other_cols]
             train_util = train_real_df[util_cols]
             test_util = test_real_df[util_cols]
             
             
             utility_message = "Synthetic Data Utility Report"
             utility_mess_text = ("<h1 style='color: #008080; font-weight:bold;'>" + utility_message + "</h1>"
                                   "<span style='color: black; font-weight:bold; font-size: 16pt'> Synthesizer: </span>"
                                   f"<span style='color: black; font-size: 14pt;'>{synthesizer}</span><br>"
                                   f"<span style='font-size: 11pt;'>{seed_info()}</span>")
             utility_mess_html = widgets.HTML(value=utility_mess_text)
             
             # --- Comparison of Real and Synthetic Dataset Summary Statistics ---
             stats_title = """<div style='color: darkblue; font-weight:bold; font-size: 16pt; margin-top: 10px; margin-bottom: 10px;'> Summary Statistics
                              <span style='color: darkblue; font-weight:bold; font-size: 16pt;'> (Real vs. Synthetic Datasets) </span></div>"""
             stats_title_html = widgets.HTML(value=stats_title)
             display(utility_mess_html, stats_title_html)
            
             continuous_vars = [var[0] for var in cont_var]
             categorical_vars = [var[0] for var in cat_var + ord_var]
                      
             # Summaries of variables per dataset                         
             if continuous_vars:
                 real_cont = util.summarize_continuous(train_util, continuous_vars, 'Real')
                 syn_cont = util.summarize_continuous(synthetic_data, continuous_vars, 'Synthetic')
                 cont_summary = pd.merge(real_cont, syn_cont, on=['Variable', 'Statistic / Category'], how='outer')
                 cont_divider = pd.DataFrame({'Variable': ['** Continuous (numerical) **'], 'Statistic / Category': [""], 'Real': [""], 'Synthetic': [""]})
             else:
                 cont_summary = pd.DataFrame()
                 cont_divider = pd.DataFrame()

             if categorical_vars:
                 real_cat = util.summarize_categorical(train_util, categorical_vars, 'Real')
                 syn_cat = util.summarize_categorical(synthetic_data, categorical_vars, 'Synthetic')
                 cat_summary = pd.merge(real_cat, syn_cat, on=['Variable', 'Statistic / Category'], how='outer')
                 cat_divider = pd.DataFrame({'Variable': ['** Categorical (nominal/ordinal) **'], 'Statistic / Category': [""], 'Real': [""], 'Synthetic': [""]})
             else:
                 cat_summary = pd.DataFrame()
                 cat_divider = pd.DataFrame()
           
             # Generate summary table
             summary_df = pd.concat([cont_divider, cont_summary, cat_divider, cat_summary], ignore_index=True).fillna("")
             summary_table, row_colors = util.prepare_table_grouped_background(summary_df)
             table_fig = util.plot_summary_table(summary_table, row_colors)
             
             # Save Summary statistics table    
             utility_dir = res_dir+"utility/"             
             if os.path.exists(utility_dir) == False:
                 os.makedirs(utility_dir)
                 
             summary_stats_filename = "real_synth_summary_stats.html"
             table_fig.write_html(utility_dir+summary_stats_filename)
             
             display(table_fig)
                        
             # --- Downstream Model Training and Testing ---
             model_eval_title = "<div style='color: darkblue; font-weight:bold; font-size: 16pt;'> Downstream Model Training and Evaluation</div>"
             model_eval_title_html = widgets.HTML(value=model_eval_title)
             model_eval_text_html = widgets.HTML(value="""<div style="font-size: 12pt; line-height: 1.5;">
                                    Train on Real data and evaluate on Real data (TRTR) <br>
                                    Train on Synthetic data evaluate on the Real dataset (TSTR).</div><br>""")   
             
             display(model_eval_title_html, model_eval_text_html)
             
             target_name = train_util.columns[-1] # last column as default target variable
             target = train_util[target_name]
             
             options, cont_names, ord_names, cat_names = util.get_estimator_options(train_util, target_name, cont_var, ord_var, cat_var)
                 
             # Input Target variable
             target_label = widgets.Label(value="Target:", layout=widgets.Layout(width='40px'))
             target_widget = widgets.Dropdown(options=train_util.columns.tolist(), value=target_name, layout=widgets.Layout(width='250px')) 
             
             # Widget to select estimator             
             estimator_widget = widgets.Dropdown(options=options, description='Estimator:', layout=widgets.Layout(width='300px')) 
             estimator_button = widgets.Button(description='Submit', disabled=False, button_style='', tooltip='submit', icon='check', layout=widgets.Layout(width='100px'))
             target_estimator_hbox = widgets.HBox([target_label, target_widget, estimator_widget, estimator_button], layout=widgets.Layout(width='80%', margin='0px 0px 25px 0px')) 
             display(target_estimator_hbox, output6)
             
             target_widget.observe(lambda change: update_classifier_options(change['new']), names='value')
             estimator_button.on_click(train_model)


# -------- Fidelity Report --------------------
def fidelity_btn_clicked(button):
    output8.clear_output()
    with output8:
        # Specify the fidelity_report sections        
        fidelity_report_sections = [
                                    # Synthesizer section
                                    {
                                        "type": "text",
                                        "sub_title": "Synthesizer:",
                                        "content": f"""<div style="font-size: 18px; margin-top:5px; margin-bottom:5px;">{synthesizer}</div>
                                                       <div style="font-size: 13px; margin-bottom:10px;">{seed_info()}</div>""",
                                    },
                                    # Univariate section
                                    {
                                        "type": "section",
                                        "title": "Univariate Resemblance",
                                                                              
                                    },
                                    ] 
        
        if uni_cont_figure_filename != "":
            uni_cont_sect = {
                                "type": "section",                               
                                "content": f"""<div style="font-size: 18px; margin-top:5px; margin-bottom:5px;"><b>Continuous Variables</b></div>""",
                                "html_files": [uni_cont_figure_filename, uni_cont_metrics_tab_filename],                                                                               
                            }
            fidelity_report_sections.append(uni_cont_sect)

        if uni_cat_figure_filename != "":
            uni_cat_sect =  {
                                "type": "section",                                        
                                "content": f"""<div style="font-size: 18px; margin-top:5px; margin-bottom:5px;"><b>Categorical Variables</b></div>""",
                                "html_files": [uni_cat_figure_filename, uni_cat_metrics_tab_filename],
                                "page_break": True,
                            }
            fidelity_report_sections.append(uni_cat_sect)

        # Bivariate section 
        bivariate_sect = {
                            "type": "section",
                            "content": f"""<div style="color:darkblue; font-size:24px; font-weight:bold;">Bivariate Resemblance</div>""",                                                                 
                         }
        fidelity_report_sections.append(bivariate_sect)
      
    
        if  bi_combined_spearman_figs != "":
             bi_cont_ord_sect = {
                                    "type": "section",                               
                                    "content": f"""<div style="font-size: 18px; margin-top:5px; margin-bottom:10px;"><b>Continuous and Ordinal Variables</b></div>
                                                    <div style="font-size: 18px;"> <span style="color:#0f736b; font-weight:bold;"> Real(Train) vs Synthetic:</span> Pairwise Correlation</div>""",
                                                    
                                    "html_files": [bi_combined_spearman_figs], 
                                 }
                 
             fidelity_report_sections.append(bi_cont_ord_sect)    
                               
             bi_forbenius_sect = {  
                                    "type": "section",                
                                    "content": f"""<div style="margin-top:20px;"><p><b>Frobenius Norm</b> = {frobenius_norm} <br>
                                               Computed from the difference between the Spearman's correlation matrices</p></div>""",
                                 }
            
             fidelity_report_sections.append(bi_forbenius_sect)

        if bi_combined_cramer_figs != "":
            bi_nominal_ord_sect = {
                                    "type": "section",                               
                                    "content": f"""<div style="font-size:18px; margin-top:5px; margin-bottom:10px;"><b>Nominal and Ordinal Variables</b></div>
                                                    <div style="font-size:18px;"> <span style="color:#0f736b; font-weight:bold;"> Real(Train) vs Synthetic:</span> Pairwise Correlation</div>""",                                                            
                                    "html_files": [bi_combined_cramer_figs], 
                                 }
            fidelity_report_sections.append(bi_nominal_ord_sect)

            bi_contingency_simil_sect = {
                                        "type": "section",                               
                                        "content": f"""<div style="font-size:18px;"> <span style="color:#0f736b; font-weight:bold;"> 
                                                    Real(Train) vs Synthetic:</span> Two-way Contingency</div>""",
                                                        
                                        "html_files": [bi_contingency_simil_file], 

                                        "page_break": True,
                                         }
            fidelity_report_sections.append(bi_contingency_simil_sect)

        # Multivariate section 
        multivariate_sect = {
                                "type": "section",
                                "title": "Multivariate Resemblance",                                                                 
                             }
        fidelity_report_sections.append(multivariate_sect)

        if multi_combined_prop_distr_figs != "":
            multi_propensity_sect = {
                                    "type": "section",
                                    "content": f""" <span style= "margin-top:20px; margin-bottom: 10px; font-size:16pt; color:#0f736b; font-weight:700;">
                                                    Propensity Score based on Logistic Regression                                                
                                                    </span>
                                                    <ul style='color:black; font-size:12pt; margin-top:5px; padding-left:18px;'>
                                                        <li>In-sample model <b>AUC</b> = {pAUC}</li>
                                                        <li>Propensity score <b>MSE</b> = {pMSE}
                                                            <i class='fa fa-info-circle' 
                                                               style='font-size:12px; color:#bcc0c0; background:white; border-radius: 50%; padding: 2px; cursor: pointer;'
                                                               title="Estimate Propensity Score Mean Square Error using equation (1) in Mi-Ja Woo 2009"></i>
                                                        </li>
                                                    </ul>""",
            
                                    "html_files": [multi_combined_prop_distr_figs],   
                                    }
            fidelity_report_sections.append(multi_propensity_sect)

        # Data Labeling section
        if multi_data_labeling_table_file !="":
            multi_data_labeling_sect = {
                                        "type": "section",
                                        "content": f"""<div style= "margin-top:20px; margin-bottom:10px; font-size:16pt; color:#0f736b; font-weight:700;">
                                                        Data Labeling Analysis </div>
                                                        <div style ='font-size:14px; margin-top:5px; margin-bottom:10px;'>Metrics are the mean values from 5-fold cross-validation.</div>""",
                                        "html_files": [multi_data_labeling_table_file],  
                                        }
            fidelity_report_sections.append(multi_data_labeling_sect)
            # Note directly below the table
            fidelity_report_sections.append({
                                        "type": "section",
                                        "content": f"""<div style='font-size:14px; line-height:1.5; margin:0;'>{DLA_NOTE}</div>""",
                                        })
            

        # Dimensionality reduction (PCA and t-SNE) section
        if multi_combined_pca_tsne_figs !="":
            multi_dimension_reduction_sect = {
                                                "type": "section",
                                                "content": f"""<div style='margin-top:20px; margin-bottom:5px; color:#0f736b; font-weight:700; font-size:16pt;'> 
                                                            Visualization with Dimensionality Reduction Techniques 
                                                            </div>
                                                            <div style="font-size: 18px; margin-top:5px; margin-bottom:5px;"><b>Continuous Variables</b></div>
                                                            """,                
                                                "html_files": [multi_combined_pca_tsne_figs],   
                                             }
            fidelity_report_sections.append(multi_dimension_reduction_sect)
            

        # UMAP section 
        if multi_umap_fig !="":
            multi_umap_sect = {
                                "type": "section",
                                "content": f"""<div style="font-size:18px; margin-top:5px; margin-bottom:5px;">
                                            <b>All Variables</b></div>
                                            """,                
                                "html_files": [multi_umap_fig],   
                                }
            fidelity_report_sections.append(multi_umap_sect) 
          
        fidelity_report = util.build_report(input_dir= fidelity_dir, sections=fidelity_report_sections, 
                                                      output_file= res_dir+"fidelity_report.html", title= "Synthetic Data Fidelity Report", 
                                                      title_style={"color": "#008080", "font-weight": "bold",})
                     
        fidelity_report_text_html = widgets.HTML(value=f"""<div style="color:#800080; font-weight:bold; font-size: 11pt; margin-top:15px;">Fidelity report saved in:</b> {fidelity_report} </div>""")   
        display(fidelity_report_text_html)
       

# ------- Privacy Report ---------------------

AUC_EXPLANATION = (
    "<b>AUC</b> indicates how well the distance to the synthetic data ranks training members above non-members <br>"
    "(1.0 = every member is closer than every non-member; 0.5 = no better than random guessing; "
    "< 0.5 = non-members are closer)."
)

def min_dcr_value_change(change):
    output5.clear_output()   
    global as_dcr_matches, k_input, privacy_dir, min_dcr, DRC_TT_ST_fig_file,\
           DCR_absolute_metrics_tab, DCR_relative_metrics_tab, DCR_attack_synth_distr_file, \
           DCR_attack_synth_auc_table, pk_table_filename, n_datasets, auc_note
    
    with output5: 
        
        min_dcr = min_dcr_float.value      
        # -- DCR Absolute Metrics--
        
        # Train-to-Train (TTD)
        tt_dcr_matches, tt_min_dcr_matched = util.record_matching(tt_distances, tt_closest_idx, min_dcr, labels=["Train_index", "Train1_index", "TTD"])        
        # Compute TTD Matching Rate
        tt_mr = tt_min_dcr_matched.shape[0]/train_real_df.shape[0]
        tt_dcr_median = np.median(tt_dcr_matches["TTD"]) 

        # Train-to-Holdout (THD) 
        th_dcr_matches, th_min_dcr_matched = util.record_matching(th_distances, th_closest_idx, min_dcr, labels=["Train_index", "Holdout_index", "THD"])
        # Compute THD Matching Rate
        th_mr = th_min_dcr_matched.shape[0]/train_real_df.shape[0]
        th_dcr_median = np.median(th_dcr_matches["THD"])
      
        # Train-to-Synthetic (TSD) 
        ts_dcr_matches, ts_min_dcr_matched = util.record_matching(ts_distances, ts_closest_idx, min_dcr, labels=["Train_index", "Synthetic_index", "TSD"])
        # Compute TSD Matching Rate
        ts_mr = ts_min_dcr_matched.shape[0]/train_real_df.shape[0]
        ts_dcr_median = np.median(ts_dcr_matches["TSD"])
       
        # Holdout-to-Synthetic (HSD)
        hs_dcr_matches, hs_min_dcr_matched = util.record_matching(hs_distances, hs_closest_idx, min_dcr, labels=["Holdout_index", "Synthetic_index", "HSD"])       
        # Compute SHD Matching Rate
        hs_mr = hs_min_dcr_matched.shape[0]/test_real_df.shape[0]
        hs_dcr_median = np.median(hs_dcr_matches["HSD"])
                                 
        # DCR Distribution
        #print()
        dcr_hist_mess = "<span style='color:black; font-weight:600; font-size: 13pt'> DCR Distribution </span>"
        dcr_hist_mess_html = widgets.HTML(value=dcr_hist_mess, layout=widgets.Layout(margin='0px 0px 0px 0px'))
        
        dcr_tt = tt_dcr_matches["TTD"]
        dcr_ts = ts_dcr_matches["TSD"]     
       
        # Compute DCR histograms manually, using plotly Bar instead of Histogram for better comparison.
        max_dcr_tt = np.max(dcr_tt)
        max_dcr_ts = np.max(dcr_ts)
        
        # Auto-scale the histograms to a more meaningful data range
        max_range = max(max_dcr_tt, max_dcr_ts)
        buffer = 0.01 # to ensure the last point is included
        max_edge = min(1.0, max_range + buffer) # Gower distance is capped at 1
        num_bins = 30       
        bins = np.linspace(0, max_edge, num_bins + 1)  # num_bins edges = num_bins bars
        
        tt_hist, _ = np.histogram(dcr_tt, bins=bins)
        ts_hist, _ = np.histogram(dcr_ts, bins=bins)

        # Compute bin centers for bar placement
        bin_centers = (bins[:-1] + bins[1:]) / 2
        bin_width = np.diff(bins)[0]
                        
        distances_fig = go.Figure()
        distances_fig.add_trace(go.Bar(x=bin_centers, y=tt_hist, width=bin_width, name="Train-to-Train", marker_color="#333F44", opacity=0.5))
        distances_fig.add_trace(go.Bar(x=bin_centers, y=ts_hist, width=bin_width, name="Train-to-Synthetic", marker_color="#37AA9C", opacity=0.5))
        distances_fig.add_trace(go.Scatter(x=[min_dcr, min_dcr], y=[0, max(max(tt_hist), max(ts_hist))], 
                                           mode='lines', name=f'Minimum DCR = {min_dcr}', line=dict(color='red', dash='dash')))     
                       
        distances_layout = go.Layout(xaxis_title="Gower's Distance", xaxis=dict(range=[bins[0], bins[-1]]), yaxis_title='Count', 
                                     barmode='overlay', legend=dict(x=1, y=0.93, xanchor='right', yanchor='top'), 
                                     width=650, height=400, margin=dict(l=0, r=0, t=0, b=10), template='plotly_white')
        
        distances_figw = go.FigureWidget(data=distances_fig.data, layout=distances_layout) 
        
        # Create privacy directory to save figures and tables for the privacy report    
        privacy_dir = res_dir+"privacy/"             
        if os.path.exists(privacy_dir) == False:
            os.makedirs(privacy_dir)

        DRC_TT_ST_fig_file = "DCR_TT_ST_distributions.html"
        distances_figw.write_html(privacy_dir+DRC_TT_ST_fig_file)        
        display(dcr_hist_mess_html, distances_figw)                     
                          
        # Create Match table        
        if min_dcr == 0.0:
            dist_label = '<b> Records with DCR = '+str(min_dcr)+' (Exact Matches)</b>'
        else:
            dist_label = '<b> Records with DCR <= '+str(min_dcr)+'</b>'
            
        match_tab_header = ['<b>DCR Absolute Metric</b>', '<b>Median DCR</b>', dist_label]
        match_rate = [['<b>Train-to-Train (TTD)</b>', '<b>Train-to-Synthetic (TSD)</b>','<b>Holdout-to-Synthetic (HSD)</b>'],
                      [str(np.round(tt_dcr_median, decimals=3)), str(np.round(ts_dcr_median, decimals=3)), str(np.round(hs_dcr_median, decimals=3))],
                      [str(tt_min_dcr_matched.shape[0])+' ('+str(np.round(tt_mr*100, decimals=1))+'%)', 
                       str(ts_min_dcr_matched.shape[0])+' ('+str(np.round(ts_mr*100, decimals=1))+'%)', 
                       str(hs_min_dcr_matched.shape[0])+' ('+str(np.round(hs_mr*100, decimals=1))+'%)']]
        
        match_tab = go.FigureWidget(data=[go.Table(header=dict(values=match_tab_header, align='left'), cells=dict(values=match_rate, height=25, align='left'), columnwidth=[220, 120, 310])])  
        match_tab.update_layout(height=180, width=650, margin=dict(l=0, r=0, t=20, b=20)) 

        DCR_absolute_metrics_tab = "DCR_absolute_metrics_table.html"

        match_tab.write_html(privacy_dir+ DCR_absolute_metrics_tab)
        
        match_tab_grid = widgets.GridBox(children=[match_tab], layout=widgets.Layout(
                      grid_template_columns= 'auto',
                      grid_template_rows= 'auto', 
                      grid_gap='0px 0px',  
                      align_item='top',
                      height='180px',
                      width='650px'))
        
        display(match_tab_grid)
               
        # DCR Relative metrics        
        # Vulnerable records are those where TSD = 0 and THD !=0
        TSD_THD_df = ts_dcr_matches.merge(th_dcr_matches, on=["Train_index"])      
        TSD_THD_df["VR_eq"] = ((TSD_THD_df["TSD"] == 0.0) & (TSD_THD_df["THD"] != 0.0)).astype(int)
        VR_eq = TSD_THD_df["VR_eq"].sum()
    
        # Vulnerable records are those where TSD < THD  
        TSD_THD_df["VR_le"] = (TSD_THD_df["TSD"] < TSD_THD_df["THD"]).astype(int)
        VR_le = TSD_THD_df["VR_le"].sum()   
                      
        risk_tab_header = ['<b>DCR Relative Metric</b>', '<b>Records</b>']
        risk_records = [['<b>TSD = 0 and THD != 0 </b>', '<b>TSD < THD </b>'],
                       [str(VR_eq)+' ('+str(np.round(VR_eq/TSD_THD_df.shape[0]*100, decimals=1))+'%)',
                        str(VR_le)+' ('+str(np.round(VR_le/TSD_THD_df.shape[0]*100, decimals=1))+'%)']]
        
        risk_tab = go.FigureWidget(data=[go.Table(header=dict(values=risk_tab_header, align='left'), 
                                                  cells=dict(values=risk_records, height=25, align='left'), columnwidth=[310, 310])])        
        risk_tab.update_layout(height=130, width=650, margin=dict(l=0, r=0, t=0, b=0)) 

        DCR_relative_metrics_tab = "DCR_relative_metrics_table.html"
        risk_tab.write_html(privacy_dir+ DCR_relative_metrics_tab)
        
        risk_tab_grid = widgets.GridBox(children=[risk_tab], layout=widgets.Layout(
                      grid_template_columns= 'auto',
                      grid_template_rows= 'auto', 
                      grid_gap='0px 0px',  
                      align_item='top',
                      height='130px',
                      width='650px'))
        
        display(risk_tab_grid)   

        # Membership Disclosure Vulnerability        
        member_disclosure_mess =  "<span style='color:darkblue; font-weight:bold; margin-top:10px; font-size: 16pt'> Membership Disclosure Vulnerability</span>"
        member_disclosure_mess_html = widgets.HTML(value=member_disclosure_mess)      
        attack_text = """
        <span style='color:#0f736b; font-weight:700; margin-top:10px; font-size: 14pt'> Adversary DCR Attack</span><br>
        <ul style='font-size: 11pt'>
            <li>The adversary has access to a synthetic dataset that has been generated using the training dataset.</li>
            <li>The adversary has an attack dataset composed of 50% records from the training dataset and 50% records not seen during training.</li>            
            <li>The adversary’s goal is to determine which records in the attack dataset were members of the training dataset.</li>
        </ul>
        """        
        attack_html_widget = widgets.HTML(value=attack_text)
        display(member_disclosure_mess_html, attack_html_widget)
        
        # Sample Attacker's dataset (same size as the training set with 50% members and 50% non-members)
        members, non_members, attack_df, labels = util.sample_attack_df(train_priv, test_priv, state=SEED)     
        
        # Gower's DCR between Attack dataset and Synthetic dataset
        asd_spinner = widgets.IntProgress(value=0, min=0, max=0, description="Computing Attack-to-Synthetic DCR...", 
                                                  style={'description_width': '250px'}, layout=widgets.Layout(width='500px'))
        display(asd_spinner)                
        as_distances, as_closest_idx = metrics.gower_distances(attack_df, synth_priv, params=gower_ref)
        asd_spinner.close()
                
        as_dcr_matches, as_min_dcr_matched = util.record_matching(as_distances, as_closest_idx, min_dcr, labels=["Attack_index", "Synthetic_index", "ASD"])
        as_dcr_matches["ground_truth"] = labels               

        # Histograms of Attack-to-Synthetic DCR distribution for Members and Non-members.
        asd_html_widget = widgets.HTML(value="<div style='color:black; font-weight:600; font-size:13pt'>Attack-to-Synthetic DCR Distribution</div>") 
        
        pos_members = as_dcr_matches[as_dcr_matches["ground_truth"] == 1]
        neg_members = as_dcr_matches[as_dcr_matches["ground_truth"] == 0]        
        max_asd_pos = np.max(pos_members["ASD"])
        max_asd_neg = np.max(neg_members["ASD"])
        
        # Auto-scale the histograms to a more meaningful data range
        max_asd_range = max(max_asd_pos, max_asd_neg)
        buffer = 0.01 # to ensure the last point is included
        max_asd_edge = min(1.0, max_asd_range + buffer) # Gower distance is capped at 1 
        asd_num_bins = 30
        asd_bins = np.linspace(0, max_asd_edge, asd_num_bins + 1)  # num_bins edges = num_bins bars
        
        pos_hist, _ = np.histogram(pos_members["ASD"], bins=asd_bins)
        neg_hist, _ = np.histogram(neg_members["ASD"], bins=asd_bins)

        # Compute bin centers for bar placement
        asd_bin_centers = (asd_bins[:-1] + asd_bins[1:]) / 2
        asd_bin_width = np.diff(asd_bins)[0]
                        
        asd_fig = go.Figure()
        asd_fig.add_trace(go.Bar(x=asd_bin_centers, y=pos_hist, width=asd_bin_width, name="Members", marker_color="#333F44", opacity=0.5))
        asd_fig.add_trace(go.Bar(x=asd_bin_centers, y=neg_hist, width=asd_bin_width, name="Non-members", marker_color="#37AA9C", opacity=0.5))  
                       
        asd_layout = go.Layout(xaxis_title="Gower's Distance", xaxis=dict(range=[asd_bins[0], asd_bins[-1]]), yaxis_title='Count', 
                                     barmode='overlay', legend=dict(x=1, y=0.93, xanchor='right', yanchor='top'), 
                                     width=650, height=400, margin=dict(l=0, r=0, t=0, b=0), template='plotly_white')
        
        asd_figw = go.FigureWidget(data=asd_fig.data, layout=asd_layout)

        DCR_attack_synth_distr_file = "DCR_attack_synth_distrib.html"
        asd_figw.write_html(privacy_dir+DCR_attack_synth_distr_file)
        display(asd_html_widget, asd_figw)

        # Mann-Whitney test for the attack rule "closer to the synthetic data = member"
        # Scores are negated distances, so higher score = closer
        member_scores = -pos_members["ASD"].to_numpy()
        non_member_scores = -neg_members["ASD"].to_numpy()

        # One-sided: H1 = members are closer than non-members (membership leakage)
        u_stat, p_value = stats.mannwhitneyu(member_scores, non_member_scores, alternative='greater')
        n_pos, n_neg = len(member_scores), len(non_member_scores)
        auc = u_stat / (n_pos * n_neg) 
               
        p_format = util.pvalue_format(p_value)
        auc_r = round(auc, 2) 

        if auc_r > 0.5 and p_value < 0.05:
            if auc_r >= 0.7:
                auc_note = ("Members are clearly closer to the synthetic data than non-members: strong indication of membership leakage. <br> "
                            "Do not share the synthetic data without further privacy measures (e.g., a DP synthesizer or a lower ε).")
            elif auc_r >= 0.6:
                auc_note = ("Members are closer to the synthetic data than non-members: possible membership leakage.<br> "
                            "Review before sharing the synthetic data.")
            else:
                auc_note = ("Members are slightly closer to the synthetic data than non-members "
                            "(statistically significant, but a weak signal for an attacker).")
        elif auc_r < 0.5:
            auc_note = ("Non-members are closer to the synthetic data than members (AUC < 0.5).<br>"
                        "The attack performs no better than random guessing.")
        else:
            auc_note = "No significant difference between members and non-members."

       
        mw_header = ["<b>AUC</b>", "<b>MW p-value (one-sided)</b>"]
        mw_values = [f"{auc_r:.2f}", p_format]
        
        mw_tab = go.FigureWidget(data=[go.Table(header=dict(values=mw_header, align='left', height=28), 
                                                columnwidth=[100, 100], cells=dict(values=mw_values, align='left', height=25))])  
        mw_tab.update_layout(width=400, height=75, margin=dict(l=0, r=0, t=10, b=0))
        
        DCR_attack_synth_auc_table = "DCR_attack_synth_auc_table.html"
        mw_tab.write_html(privacy_dir+DCR_attack_synth_auc_table)
                    
        mw_tab_grid = widgets.GridBox(children=[mw_tab], layout=widgets.Layout(
                      grid_template_columns= 'auto',
                      grid_template_rows= 'auto', 
                      grid_gap='0px 0px',  
                      align_item='flex-start',
                      height='75px',
                      margin='0px'))

        auc_expl_html = widgets.HTML(
            f"<div style='font-size:11pt; line-height:1.5;'><br>{AUC_EXPLANATION}</div>")

        auc_note_html = widgets.HTML(
            f"<div style='color:darkblue; font-size:10pt; line-height:1.5; margin:0px 0px;'>{auc_note}</div>",
            layout=widgets.Layout(margin='0px 0px'))

        display(widgets.VBox([auc_expl_html, mw_tab_grid, auc_note_html],
                             layout=widgets.Layout(margin='0px 0px 10px 0px')))

        # Precision@K for membership inference
        precision_k_text = """
        <div style='color:black; font-weight:600; margin-top:5px; font-size: 13pt;'> Precision at K (Precision@K)</div> 
        <div style ='font-size: 11pt; line-height: 1.5;'>The attack dataset is ranked by ascending Attack-to-Synthetic DCR values to assess <br> 
        how many training members appear among the top-K most similar data points.</div>     
        """        
        precision_k_html_widget = widgets.HTML(value=precision_k_text)
        display(precision_k_html_widget)

        y_true = as_dcr_matches["ground_truth"].to_numpy()
        y_scores = as_dcr_matches["ASD"].to_numpy() 
        k_row = [f"<b>K</b>"]
        pk_row = [f"<b>Expected members</b>"]
        for k in np.arange(1, 11):
            precision_at_k = metrics.precision_at_k_low_scores(y_true, y_scores, k)
            k_row.append(f"<b>{k}</b>")
            num_members = precision_at_k * k
            pk_row.append(f"{num_members:.0f}")

        # Generate attack datasets
        n_datasets= 100 
        if train_real_df.shape[0] > 1000:
            n_datasets = 10           
        
        progress = widgets.IntProgress(value=0, min=0, max=n_datasets, description='Progress:', bar_style='info')
        display(progress)
        
        attack_datasets = [(y_true, y_scores)]
        for i in np.arange(2, n_datasets+1):   
            members_i, non_members_i, attack_df_i, labels_i = util.sample_attack_df(train_priv, test_priv, state=SEED+i)
            as_distances_i, as_closest_idx_i = metrics.gower_distances(attack_df_i, synth_priv, params=gower_ref)
            as_dcr_matches_i, as_min_dcr_matched_i = util.record_matching(as_distances_i, as_closest_idx_i, labels=["Attack_index", "Synthetic_index", "ASD"])
            as_dcr_matches_i["ground_truth"] = labels_i
            y_true_i = as_dcr_matches_i["ground_truth"].to_numpy()
            y_scores_i = as_dcr_matches_i["ASD"].to_numpy()
            attack_datasets.append((y_true_i, y_scores_i))
            progress.value = i

        progress.close()
        
        # Compute average precision@k for k = 1 to 10 
        k_values = range(1, 11)
        avg_precisions = [f"<b>Avg. Precision@K</b>"]        
        for k in k_values:
            precisions = []
            for y_true, y_scores in attack_datasets:
                p_at_k = metrics.precision_at_k_low_scores(y_true, y_scores, k)                
                precisions.append(p_at_k)
            avg_precisions.append(f"{np.mean(precisions):.2f}")
            
        pk_tab_rows = list(zip(*[pk_row, avg_precisions]))
            
        # Table for precision@k with k values 1 to 10 
        col_widths = [150, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50]
        pk_tab = go.FigureWidget(data=[go.Table(header=dict(values=k_row, align='left'), 
                                                cells=dict(values=pk_tab_rows, height=25, align='left'), columnwidth=col_widths)])  
        pk_tab.update_layout(height=120, width=650, margin=dict(l=0, r=0, t=15, b=15))
        
        pk_table_filename = "precision_at_k_table.html"
        pk_tab.write_html(privacy_dir+pk_table_filename)
        
        pk_tab_grid = widgets.GridBox(children=[pk_tab], layout=widgets.Layout(
                      grid_template_columns= 'auto',
                      grid_template_rows= 'auto', 
                      grid_gap='0px 0px',  
                      align_item='top',
                      height='120px',
                      width='650px'))
        
        display(pk_tab_grid) 
        
        avg_pk_text = f"""
                        <div style="font-size: 11pt; line-height: 1.5;">
                        The <b>Avg. Precision@K</b> is calculated across <b>{n_datasets}</b> attacker datasets 
                        to show how frequently, on average, <br> 
                        training members appear among the top-K.
                        </div>
                        """
        pk_expl_widget = widgets.HTML(value=avg_pk_text)        
        avg_pk_formula = widgets.HTMLMath(value=r"""<div style="font-size: 11pt;">
                                                <p>\[\text{Average Precision@K} = \frac{1}{N} \sum_{i=1}^{N} \text{Precision@K}_i\]</p>
                                                <p> Where:<br> \(N\) = number of datasets </p></div>""")
        display(pk_expl_widget, avg_pk_formula)
        
        # Button for saving the privacy report
        print()
        privacy_report_btn = widgets.Button(
                    description='Save Privacy Report',
                    disabled=False,
                    button_style='', 
                    tooltip='Save Privacy Report',
                    icon='check',
                    layout=widgets.Layout(width='200px'),
                    style={'font_size': '14px'})
        privacy_report_btn.on_click(privacy_btn_clicked)
        display(privacy_report_btn, output9)

def privacy_btn_clicked(button):
    output9.clear_output()
    with output9:
        # Specify the utility_report sections 
        privacy_report_sections = [
                                    # First section
                                    {
                                        "type": "text",
                                        "sub_title": "Synthesizer:",
                                        "content": f"""<div style="font-size: 18px; margin-top:5px; margin-bottom:5px;">{synthesizer}</div>
                                                       <div style="font-size: 13px; margin-bottom:10px;">{seed_info()}</div>""",
                                    },
                                    # Second section
                                    {
                                        "type": "section",
                                        "title": "Distance to Closest Record (DCR)",        
                                        "content": f"""<div style='font-size: 11pt;'>
                                                   DCR measures the similarity between records in two datasets, for example, between Train (Real) and Synthetic records.<br>
                                                   A low DCR alone <b>does not imply identity disclosure</b>.
                                                   It should be interpreted as a proximity indicator, not evidence of a privacy breach.</div>
                                                    """, 
                                    },   
                                    # Third section
                                    {
                                        "type": "section",
                                        "sub_title": "Record Matching",
                                        "content": f"""<div style="font-size: 14px; margin-top:5px; margin-bottom:5px;"><b>DCR Distribution</b>""",
                                        "html_files": [DRC_TT_ST_fig_file, DCR_absolute_metrics_tab, DCR_relative_metrics_tab ],                                    
                                    },
            
                                    # Fourth section: attack description + DCR distribution
                                    {
                                        "type": "section",
                                        "title": "Membership Disclosure Vulnerability",
                                        "sub_title": "Adversary DCR Attack",
                                        "content": f"""<div>
                                                            <ul style='font-size: 11pt;'>
                                                                <li>The adversary has access to a synthetic dataset that has been generated using the training dataset.</li>
                                                                <li>The adversary has an attack dataset composed of 50% records from the training dataset and 50% records not seen during training.</li>
                                                                <li>The adversary’s goal is to determine which records in the attack dataset were members of the training dataset.</li>
                                                            </ul>
                                                            <div style="font-size: 14px; margin-top:5px; margin-bottom:5px;"><b>Attack-to-Synthetic DCR Distribution</b></div>
                                                    </div>""",
                                        "html_files": [DCR_attack_synth_distr_file],
                                    },
                                    # AUC explanation + AUC table
                                    {
                                        "type": "section",
                                        "content": f"""<div style='font-size:11pt; line-height:1.5;'>{AUC_EXPLANATION}</div>""",
                                        "html_files": [DCR_attack_synth_auc_table],
                                    },
                                    # AUC interpretation, directly below the table
                                    {
                                        "type": "section",
                                        "content": f"""<div style='color:darkblue; font-size:11pt; line-height:1.5; margin:0;'>{auc_note}</div>""",
                                    },
            
                                    # Fifth section
                                    {
                                        "type": "section",                                       
                                        "content": f"""
                                                    <div style='color:black; font-weight:600; margin-top:5px; font-size: 13pt;'> Precision at K (Precision@K)</div> 
                                                    <div style ='font-size: 11pt; line-height: 1.5;'>The attack dataset is ranked by ascending Attack-to-Synthetic DCR values to assess <br> 
                                                    how many true training members appear among the top-K most similar data points.</div>     
                                                    """,
                                        "html_files": [pk_table_filename],                                    
                                    }, 
                                     # Sixth section
                                    {
                                        "type": "section",
                                        "content": f"""
                                                    <div style="font-size: 11pt; line-height: 1.5;">
                                                    The <b>Avg. Precision@K</b> is calculated across <b>{n_datasets}</b> attacker datasets 
                                                    to show how frequently, on average, <br> 
                                                    training members appear among the top-K.
                                                    </div>
                                                    """,
                                    },    
                                          
                                   ] 
                 
        privacy_report = util.build_report(input_dir= privacy_dir, sections= privacy_report_sections, 
                                                      output_file= res_dir+"privacy_report.html", title= "Synthetic Data Privacy Report", 
                                                      title_style={"color": "#800880", "font-weight": "bold",})
                     
        privacy_report_text_html = widgets.HTML(value=f"""<div style="color: #800080; font-weight:bold; font-size: 11pt; margin-top:15px;">Privacy report saved in:</b> {privacy_report} </div>""")   
        display(privacy_report_text_html)        

# ------- Utility Report --------------------- 
def update_classifier_options(target_name): 
    output6.clear_output()
    global cont_names, ord_names, cat_names
    with output4:        
        options, cont_names, ord_names, cat_names = util.get_estimator_options(train_real_df, target_name, cont_var, ord_var, cat_var)
        if not options:
            estimator_widget.options = []
            with output6:
                display(widgets.HTML("<span style='color:darkred; background-color:LightYellow; font-size:10pt;'>This variable cannot be used as a target "
                                     "(datetime/id). Choose a numerical or categorical variable.</span>"))
            return
        estimator_widget.options = options
        estimator_widget.value = options[0]
       

# --- Functions used in Model training and evaluation ---
def train_and_evaluate(X_train_real, y_train_real, X_synthetic, y_synthetic, X_test_real, model_cls, **kwargs):
    # TRTR (train in Real)
    model_real = model_cls(**kwargs)
    model_real.fit(X_train_real, y_train_real)
    
    # TSTR (train in synthetic)
    model_syn = model_cls(**kwargs)
    model_syn.fit(X_synthetic, y_synthetic)
      
    # Test model based on its type
    y_pred_real = []           
    y_pred_syn = []
    if is_classifier(model_real):        
        if hasattr(model_real, "predict_proba"):            
            y_pred_real = model_real.predict_proba(X_test_real)           
            y_pred_syn = model_syn.predict_proba(X_test_real)
        else:
            y_pred_real = model_real.decision_function(X_test_real)           
            y_pred_syn = model_syn.decision_function(X_test_real)
            
    elif is_regressor(model_real):
        y_pred_real = model_real.predict(X_test_real)           
        y_pred_syn = model_syn.predict(X_test_real)
        
    return y_pred_real, y_pred_syn, model_real, model_syn        

def plot_metric_results(headers, rows, target, estimator):
    global downstream_filename    
    min_width = 80   # minimum column width in pixels
    char_width = 8   # approximate pixel width per character
    colwidths = [max(min_width, len(h) * char_width) for h in headers]
    fig = go.FigureWidget(
        data=[
            go.Table(
                header=dict(values=[f"<b>{h}</b>" for h in headers], align="left"),
                cells=dict(values=list(zip(*rows)), height=25, align="left"),
                columnwidth=colwidths,
            )
        ]
    )
    fig.update_layout(width=950, height=120, margin=dict(l=0, r=0, t=0, b=0))

    utility_dir = res_dir+"utility/"
    if os.path.exists(utility_dir) == False:
        os.makedirs(utility_dir)  
    
    downstream_filename = util.safe_filename(f"downstream_train_eval_{target}_{estimator}") + ".html"
    fig.write_html(utility_dir+downstream_filename)
    
    grid = widgets.GridBox(children=[fig], layout=widgets.Layout(
            grid_template_columns="auto",
            grid_template_rows="auto",
            grid_gap="0 0",
            margin="0 0 0px 0",
            align_items="flex-start",
            height="120px",
            width="950px"))
    display(grid)
    
# --- Functions for computing and plotting SHAP values Distribution per Feature ---  
def shap_with_progress(explainer, X, message="Computing SHAP", batch_size=200, explainer_type="auto"):
    n = len(X)
    if n > 1000:
        batch_size = 500
   
    progress_bar = widgets.IntProgress( value=0, min=0, max=100, description=message, bar_style='info', 
                                        style={'description_width': '250px'}, layout=widgets.Layout(width='500px'))
    display(progress_bar)

    # Determine if we need the silent argument
    silent_flag = True if explainer_type in ["kernel", "kernel class"] else False

    # Compute SHAP values in batches
    ini_explanation = explainer(X[:batch_size], silent=silent_flag) if silent_flag else explainer(X[:batch_size])
    progress_bar.value = int((batch_size / n) * 100)

    for start in range(batch_size, n, batch_size):
        end = min(start + batch_size, n)
        batch_explanation = explainer(X[start:end], silent=silent_flag) if silent_flag else explainer(X[start:end])

        # Concatenate results
        ini_explanation.values = np.concatenate((ini_explanation.values, batch_explanation.values))
        ini_explanation.data = np.concatenate((ini_explanation.data, batch_explanation.data))
        ini_explanation.base_values = np.concatenate((ini_explanation.base_values, batch_explanation.base_values))

        # Update progress bar
        progress_bar.value = int((end / n) * 100)

    progress_bar.close()
    return ini_explanation

# --- KernelExplainer settings (KNN models) ---
SHAP_KERNEL_ROWS = 200        # test rows explained
SHAP_KERNEL_NSAMPLES = 500    # model evaluations per explained row
SHAP_KERNEL_BACKGROUND = 50   # background rows

def kernel_shap_with_progress(explainer, X, message, nsamples, batch_size=20):
    """Kernel SHAP in small batches with a progress bar. Returns a shap.Explanation with
    values (n, f) or (n, f, k) and base_values (n,) or (n, k)."""
    n = len(X)
    progress_bar = widgets.IntProgress(value=0, min=0, max=100, description=message, bar_style='info',
                                       style={'description_width': '250px'}, layout=widgets.Layout(width='500px'))
    display(progress_bar)

    parts = []
    for start in range(0, n, batch_size):
        sv = explainer.shap_values(X.iloc[start:start + batch_size], nsamples=nsamples,
                                   l1_reg=False, silent=True)
        if isinstance(sv, list):                 
            sv = np.stack(sv, axis=-1)
        parts.append(np.asarray(sv))
        progress_bar.value = int(min(start + batch_size, n) / n * 100)
    progress_bar.close()

    values = np.concatenate(parts, axis=0)
    ev = np.atleast_1d(np.asarray(explainer.expected_value, dtype=float))
    base_values = np.tile(ev, (n, 1)) if values.ndim == 3 else np.full(n, ev[0])
    return shap.Explanation(values=values, base_values=base_values,
                            data=X.to_numpy(), feature_names=list(X.columns))

def compute_shap_explainer(model, X_train, X_test, message, explainer_type="auto"):
    # Handle TreeExplainer
    if explainer_type == "tree":
        explainer = shap.TreeExplainer(model)
        return shap_with_progress(explainer, X_test, message, explainer_type="tree")

    # Handle LinearExplainer
    if explainer_type == "linear":
        explainer = shap.LinearExplainer(model, X_train)
        return shap_with_progress(explainer, X_test, message, explainer_type="linear")

    # Handle KernelExplainer (regression or classification)
    if explainer_type in ["kernel", "kernel class"]:
        np.random.seed(SEED)
        # Background: k-means summary for large datasets, otherwise a random sample
        if len(X_train) > 1000:
            background = shap.kmeans(X_train, 20)
        else:
            background = shap.sample(X_train, min(SHAP_KERNEL_BACKGROUND, len(X_train)), random_state=SEED)

        predict_fn = model.predict_proba if explainer_type == "kernel class" else model.predict
        explainer = shap.KernelExplainer(predict_fn, background)

        # Same random rows for the real and the synthetic model (fixed seed, same X_test)
        X_explain = X_test.sample(n=min(SHAP_KERNEL_ROWS, len(X_test)), random_state=SEED)
        return kernel_shap_with_progress(explainer, X_explain, message, SHAP_KERNEL_NSAMPLES)

    # Default: auto explainer
    explainer = shap.Explainer(model, X_test)
    return explainer

def shap_beeswarm_plot_widget(explainer, feature_order, x_min, x_max, beeswarm_title):
    # SHAP creates its own figure — customize and capture it
    custom_order = [explainer.feature_names.index(f) for f in feature_order]
    custom_cmap = LinearSegmentedColormap.from_list("blue_white_red", ["#2166ac", "#f7f7f7", "#8B0000"])  
    width = 10
    plot_height = max(6, len(custom_order) * 0.2)
    shap.plots.beeswarm(explainer, order=custom_order, max_display=25, show=False, 
                        plot_size=(width, plot_height), color=custom_cmap)
    fig = plt.gcf()
    ax = fig.axes[0]        
    ax.set_xlim(x_min, x_max)
    fig.suptitle(beeswarm_title, fontsize=16, fontweight='bold')
    ax.tick_params(labelsize=14)
    ax.tick_params(labelsize=16)
    
    # Save Shap beeswarm plot
    utility_dir = res_dir+"utility/"
    if os.path.exists(utility_dir) == False:
        os.makedirs(utility_dir)
        
    beeswarm_filename = util.safe_filename(beeswarm_title) + ".png"
    fig.savefig(utility_dir+beeswarm_filename, format='png', bbox_inches='tight')    
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)
    
    if plot_height > 6:
        width, height = 500, 600
    else:
        width, height = 500, 450   
        
    return widgets.Image(value=buf.getvalue(), format='png', width=width, height=height, 
                         layout=widgets.Layout(margin='0 50px 0 0')), beeswarm_filename
   

def select_shap_output(exp, target_class=None):
    """Return a single-output Explanation with values (n, f) and base_values (n,).
    3-D explanations (classification): select class `target_class` from both
    values (n, f, k) and base_values (n, k) or (k,).
    2-D explanations (regression / single-output binary): returned as is, with
    base_values flattened to (n,).
    """
    vals = np.asarray(exp.values)
    bv = np.asarray(exp.base_values, dtype=float)
    n = vals.shape[0]

    if vals.ndim == 3:
        if target_class is None: # default: last class (positive class in binary tasks)
            target_class = vals.shape[2] - 1
        vals = vals[:, :, target_class]
        if bv.ndim == 2:  # (n, k): one baseline per record and class
            bv = bv[:, target_class]
        elif bv.ndim == 1 and bv.shape[0] != n:  # (k,): one baseline per class
            bv = np.full(n, bv[target_class])
    else:
        bv = bv.reshape(-1) if bv.size == n else np.full(n, bv.reshape(-1)[0])

    return shap.Explanation(values=vals, base_values=bv,
                            data=exp.data, feature_names=exp.feature_names)


def plot_shap_beeswarm_comparison(real_shap_exp, syn_shap_exp, target_name, class_name, target_class=0):
    global real_beeswarm_filename, syn_beeswarm_filename
    feature_order = list(real_shap_exp.feature_names)

    is_class = np.ndim(real_shap_exp.values) == 3
    target_text = f"-{target_name} = {class_name}" if is_class else f"-{target_name}"

    real_shap_exp_single = select_shap_output(real_shap_exp, target_class)
    syn_shap_exp_single = select_shap_output(syn_shap_exp, target_class)

    x_min = min(real_shap_exp_single.values.min(), syn_shap_exp_single.values.min())
    x_max = max(real_shap_exp_single.values.max(), syn_shap_exp_single.values.max())

    beeswarm_real_title = "SHAP beeswarm Real" + target_text
    real_beeswarm_plot, real_beeswarm_filename = shap_beeswarm_plot_widget(
        real_shap_exp_single, feature_order, x_min, x_max, beeswarm_real_title)

    beeswarm_syn_title = "SHAP beeswarm Synthetic" + target_text
    syn_beeswarm_plot, syn_beeswarm_filename = shap_beeswarm_plot_widget(
        syn_shap_exp_single, feature_order, x_min, x_max, beeswarm_syn_title)

    return widgets.HBox([real_beeswarm_plot, syn_beeswarm_plot])


# --- Functions for computing and plotting Feature Importance based on mean(|SHAP|) values ---
def plot_shap_bar(sorted_features, sorted_means, x_min, x_max, title):
    
    # Plot SHAP bar plot for sorted features
    plot_height = max(6, len(sorted_features) * 0.2)
    fig, ax = plt.subplots(figsize=(10, plot_height))
    bars = ax.barh(sorted_features, sorted_means, color="#2166ac")
    
    # Keep only left and bottom spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(True)
    ax.spines['bottom'].set_visible(True)    
    
    # Dotted gray horizontal grid lines
    ax.yaxis.grid(True, linestyle=':', color='lightgray', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.set_xlim(x_min, x_max)
       
    # Bold title and axis text format          
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.set_xlabel("Mean(|SHAP value|)", fontsize=14)
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=16)
    
    # Add +value labels
    for i, v in enumerate(sorted_means):
        ax.text(v + 0.001, i, f"+{v:.2f}",
                va='center', fontsize=14, color="#2166ac", fontweight='bold')
        
    shap_bar_filename = util.safe_filename(title) + ".png"          
    fig.savefig(utility_dir+shap_bar_filename, format='png', bbox_inches='tight') 
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)  

    if plot_height > 6:
        width, height = 500, 600
    else:
        width, height = 500, 450    
  
    return widgets.Image(value=buf.getvalue(), format='png',  width=width, height=height), shap_bar_filename

def plot_shap_bar_comparison(real_shap_exp, syn_shap_exp, target_name, class_name, target_class=0): 
    global real_shap_bar_file, syn_shap_bar_file
    
    if real_shap_exp.values.ndim == 3:
        is_class = np.ndim(real_shap_exp.values) == 3
        target_text = f"{target_name} = {class_name}" if is_class else f"{target_name}"
        real_vals = select_shap_output(real_shap_exp, target_class).values
        syn_vals = select_shap_output(syn_shap_exp, target_class).values
    else:  # 2D
        real_vals = real_shap_exp.values
        syn_vals = syn_shap_exp.values
        target_text = f"{target_name}"
    
    # Compute mean absolute SHAP per feature for the real dataset
    real_shap_exp_values = real_vals
    real_mean_abs = np.abs(real_shap_exp_values).mean(axis=0)    
    real_feature_names = np.array(real_shap_exp.feature_names)    
    real_order = np.argsort(real_mean_abs)
    real_sorted_features = real_feature_names[real_order]
    real_sorted_means = real_mean_abs[real_order]

    # Compute mean absolute SHAP per feature for the synthetic dataset
    syn_shap_exp_values = syn_vals
    syn_mean_abs = np.abs(syn_shap_exp_values).mean(axis=0)   
    syn_feature_names = np.array(syn_shap_exp.feature_names)    
    syn_order = np.argsort(syn_mean_abs)
    syn_sorted_features = syn_feature_names[syn_order]
    syn_sorted_means = syn_mean_abs[syn_order]

    # Compute the min and max of the Mean(|SHAP value|) values across both real and synthetic datasets
    real_max = real_mean_abs.max()
    syn_max =  syn_mean_abs.max()
    x_min = 0.00
    x_max = max(real_max, syn_max)
    real_title = "Real Data Global SHAP-"+target_text
    real_bar_plot, real_shap_bar_file = plot_shap_bar(real_sorted_features, real_sorted_means, x_min, x_max, real_title)
    
    syn_title = "Synthetic Data Global SHAP-"+target_text
    syn_bar_plot, syn_shap_bar_file = plot_shap_bar(syn_sorted_features, syn_sorted_means, x_min, x_max, syn_title)
    
    real_syn_bar_figures = widgets.HBox([real_bar_plot, syn_bar_plot], layout=widgets.Layout(margin='30px 0 20px 0'))
     
    return real_syn_bar_figures

# --- Functions for generating the Utility Report ---
# name: (model class, fixed kwargs, SHAP explainer type)
ESTIMATORS = {
    "Linear Regression":         (LinearRegression,       {},                 "linear"),
    "KNN Regressor":             (KNeighborsRegressor,    {"n_neighbors": 5}, "kernel"),
    "Random Forest Regressor":   (RandomForestRegressor,  {},                 "tree"),
    "Logistic Regression":       (LogisticRegression,     {"max_iter": 500},  "linear"),
    "Multi Logistic Regression": (LogisticRegression,     {"max_iter": 500},  "linear"),
    "KNN Classifier":            (KNeighborsClassifier,   {"n_neighbors": 5}, "kernel class"),
    "Random Forest Classifier":  (RandomForestClassifier, {},                 "tree"),
}

def make_estimator(name):
    """Return (class, kwargs, explainer_type, is_classifier) for an estimator name.
    random_state=SEED is added at call time, so a seed chosen by the user is used."""
    cls, kwargs, explainer_type = ESTIMATORS[name]
    kwargs = dict(kwargs)
    if "random_state" in cls().get_params():
        kwargs.setdefault("random_state", SEED)
    return cls, kwargs, explainer_type, is_classifier(cls())
    
def train_model(change):
    output6.clear_output()
    output7.clear_output()
    output8.clear_output()
    output9.clear_output()
    output10.clear_output()
    global estimator, real_shap_exp, syn_shap_exp, target_name, class_options
    with output6: 
        target_name = target_widget.value
        estimator = estimator_widget.value  
        real_shap_exp = []
        syn_shap_exp = []
        target_domain = []
        
        # Split features and targets; drop datetime/id columns from the features
        drop_cols = [target_name] + [c for c in other_cols if c != target_name]
        X_train_real = train_real_df.drop(columns=drop_cols, errors='ignore').copy()
        y_train_real = train_real_df[target_name]
        X_test_real = test_real_df.drop(columns=drop_cols, errors='ignore').copy()
        y_test_real = test_real_df[target_name]
        X_synthetic = synthetic_data.drop(columns=drop_cols, errors='ignore').copy()
        y_synthetic = synthetic_data[target_name]

        # Make sure real and synthetic targets use the same data type
        if y_synthetic.dtype != y_train_real.dtype:
            try:
                y_synthetic = y_synthetic.astype(y_train_real.dtype)
            except (ValueError, TypeError):
                pass

        # Feature lists without the target
        cont = [c for c in cont_names if c != target_name]
        ord_feats = [c for c in ord_names if c != target_name]
        cat = [c for c in cat_names if c != target_name]

        # --- Preprocessing ---
        # Ordinal features -> ranks (same mapping in all datasets)
        if ord_feats:
            ord_specs = [(n, d, metadata_obj_dict["columns"][n].get("order_by", "alphabetical"))
                         for n, d in ord_var if n in ord_feats]
            X_train_real, X_test_real, X_synthetic = util.encode_ordinals(
                [X_train_real, X_test_real, X_synthetic], ord_specs)

        # Scale continuous and ordinal features (fit on Real (Train) only)
        num = cont + ord_feats
        if num:
            for X in (X_train_real, X_test_real, X_synthetic):
                X[num] = X[num].astype(float)
            scaler = StandardScaler().fit(X_train_real[num])
            for X in (X_train_real, X_test_real, X_synthetic):
                X[num] = scaler.transform(X[num])

        # One-hot encode nominal features with normalized labels (aligning columns)
        if cat:
            for X in (X_train_real, X_test_real, X_synthetic):
                for c in cat:
                    X[c] = X[c].map(util.norm_label)
            X_train_real = pd.get_dummies(X_train_real, columns=cat, drop_first=True, dtype=int)
            X_test_real = pd.get_dummies(X_test_real, columns=cat, drop_first=True, dtype=int)
            X_synthetic = pd.get_dummies(X_synthetic, columns=cat, drop_first=True, dtype=int)

            all_cols = sorted(set(X_train_real.columns) | set(X_test_real.columns) | set(X_synthetic.columns))
            X_train_real = X_train_real.reindex(columns=all_cols, fill_value=0)
            X_test_real = X_test_real.reindex(columns=all_cols, fill_value=0)
            X_synthetic = X_synthetic.reindex(columns=all_cols, fill_value=0)

       
        # --- Train and evaluate (one code path for every estimator) ---
        cls, kwargs, explainer_type, is_clf = make_estimator(estimator)
        target_domain = []
        real_shap_exp = syn_shap_exp = None

        if is_clf:
            # Encode labels as 0..k-1 using every label seen in any dataset (1, 1.0 and "1" are one class)
            norm = lambda y: [util.norm_label(v) for v in y]
            le = LabelEncoder().fit(norm(pd.concat([y_train_real, y_test_real, y_synthetic])))
            y_train_fit = le.transform(norm(y_train_real))
            y_test_eval = le.transform(norm(y_test_real))
            y_syn_fit = le.transform(norm(y_synthetic))
            n_classes = len(le.classes_)

            if len(np.unique(y_syn_fit)) < 2:
                display(widgets.HTML(
                    "<div style='color:darkred; background-color:LightYellow; font-size:10pt;'>"
                    f"The synthetic data contains only one class of <b>{target_name}</b>. "
                    "A classifier cannot be trained on it.</div>"))
                return
        else:
            y_train_fit, y_test_eval, y_syn_fit = y_train_real, y_test_real, y_synthetic

        y_pred_real, y_pred_syn, model_real, model_syn = train_and_evaluate(
            X_train_real, y_train_fit, X_synthetic, y_syn_fit, X_test_real, cls, **kwargs)

        if is_clf:
            # Same probability columns for both models (one per class)
            y_pred_real = metrics.align_proba(y_pred_real, model_real.classes_, n_classes)
            y_pred_syn = metrics.align_proba(y_pred_syn, model_syn.classes_, n_classes)
            binary = n_classes == 2
            m_real, metric_labels = metrics.classification_metrics(y_test_eval, y_pred_real, binary)
            m_syn, _ = metrics.classification_metrics(y_test_eval, y_pred_syn, binary)
            target_domain = le.inverse_transform(model_real.classes_)
        else:
            m_real = metrics.regression_metrics(y_test_eval, y_pred_real)
            m_syn = metrics.regression_metrics(y_test_eval, y_pred_syn)
            metric_labels = list(m_real.keys())

        headers = ["Training Data", "Testing Data"] + metric_labels
        rows = [["Real", "Real"] + [f"{v:.2f}" for v in m_real.values()],
                ["Synthetic", "Real"] + [f"{v:.2f}" for v in m_syn.values()]]
        plot_metric_results(headers, rows, target_name, estimator)

        # --- SHAP: only when both models know the same classes ---
        missing = set(model_real.classes_) - set(model_syn.classes_) if is_clf else set()
        if missing:
            display(widgets.HTML(
                "<div style='color:darkred; background-color:LightYellow; font-size:10pt;'>"
                f"The synthetic data has no records for class(es): {list(le.inverse_transform(sorted(missing)))}. "
                "The model trained on synthetic data can never predict them, so the SHAP comparison is skipped.</div>"))
        else:
            real_shap_exp = compute_shap_explainer(model_real, X_train_real, X_test_real,
                                                   "Computing Real SHAP Explainer...", explainer_type)
            syn_shap_exp = compute_shap_explainer(model_syn, X_synthetic, X_test_real,
                                                  "Computing Synthetic SHAP Explainer...", explainer_type)    
        if real_shap_exp is not None:              
           # Display SHAP Analysis 
           shap_title_html = widgets.HTML(value=f""" <div style='color: darkblue; font-weight:bold; font-size: 16pt;'>
                                                SHAP (SHapley Additive exPlanation) Values Analysis</div>""")
           display(shap_title_html)
           shap_dist_title_html = widgets.HTML(value=f""" <div style='color:#0f736b; font-weight:700; margin-top:10px; 
                                                     font-size: 14pt'>SHAP Values Distribution per Feature</div>""") 
           display(shap_dist_title_html)
           model_info = widgets.HTML(value=f"<div style='font-size: 12pt;'><b>Model:</b> {estimator}</div>")            
           target_info = widgets.HTML(value=f"<div style='font-size: 12pt;'><b>Target:</b> {target_name}</div>") 
           shap_info = [model_info, target_info]  
           shap_vbox = widgets.VBox(shap_info, layout=widgets.Layout(width='50%', margin='0px 0px 0px 0px',))
           display(shap_vbox)  
                                 
           if np.ndim(real_shap_exp.values) == 3:             
               class_options = [(name, i) for i, name in enumerate(target_domain)]
               class_label = widgets.HTML(value="<b style='color:black; font-size: 11pt;'>Target Class:</b>")
               class_selector= widgets.Dropdown(options=class_options, value=0)               
               class_name = class_options[class_selector.value][0]  
               class_selector_hbox = widgets.HBox([class_label, class_selector], layout=widgets.Layout(width='40%', margin='0px 0px 25px 0px'))               
               display(class_selector_hbox, output7)
               class_selector.observe(target_class) 
               
               with output7: 
                   # Beeswarm plot for analysing SHAP values distribution per feature
                   beeswarm_fig = plot_shap_beeswarm_comparison(real_shap_exp, syn_shap_exp, target_name, class_name, class_selector.value)                
                   display(beeswarm_fig)

                   # Analysis of feature importance
                   shap_importance_title_html = widgets.HTML(value=f""" <div style='color:#0f736b; font-weight:700; margin-top:40px; 
                                                                 font-size: 14pt'>Feature Importance (Mean(|SHAP value|)</div>""") 
                   display(shap_importance_title_html) 
                   real_syn_bar_figures = plot_shap_bar_comparison(real_shap_exp, syn_shap_exp, target_name, class_name, class_selector.value)
                   display(real_syn_bar_figures)                  
                  
           else: # For regression task           
               # Beeswarm plot for analysing SHAP values distribution per feature
               beeswarm_fig = plot_shap_beeswarm_comparison(real_shap_exp, syn_shap_exp, target_name, "None", None)
               display(beeswarm_fig)
               # Analysis of feature importance
               shap_importance_title_html = widgets.HTML(value=f"""<div style='color:#0f736b; font-weight:700; 
                                                               margin-top:40px; font-size: 14pt'>Feature Importance (Mean(|SHAP value|)</div>""") 
               display(shap_importance_title_html) 
               real_syn_bar_figures = plot_shap_bar_comparison(real_shap_exp, syn_shap_exp, target_name, "None", None)
               display(real_syn_bar_figures)

           # Button for saving the utility report
           utility_report_btn = widgets.Button(
                    description='Save Utility Report',
                    disabled=False,
                    button_style='', 
                    tooltip='Save Utility Report',
                    icon='check',
                    layout=widgets.Layout(width='200px'),
                    style={'font_size': '14px'})
           utility_report_btn.on_click(utility_btn_clicked)
           display(utility_report_btn, output10)
                
               
def target_class(change):    
        if change['type'] == 'change' and change['name'] == 'value':
            with output7:
                clear_output(wait=True)  # clear previous output
                selected_class = change['new']               
                class_name = class_options[selected_class][0] 
                
                # Beeswarm plot for analysing SHAP values distribution per feature
                beeswarm_fig = plot_shap_beeswarm_comparison(real_shap_exp, syn_shap_exp, target_name, class_name, selected_class)               
                display(beeswarm_fig) 

                # Analysis of feature importance
                shap_importance_title_html = widgets.HTML(value=f""" <div style='color:#0f736b; font-weight:700; margin-top:40px; 
                                                                font-size: 14pt'>Feature Importance (Mean(|SHAP value|)</div>""") 
                display(shap_importance_title_html) 
                real_syn_bar_figures = plot_shap_bar_comparison(real_shap_exp, syn_shap_exp, target_name, class_name, selected_class)
                display(real_syn_bar_figures)

def utility_btn_clicked(button):
    output10.clear_output()
    with output10:          
          # Specify the utility_report sections         
          utility_report_sections = [
                                     # First section
                                    {
                                        "type": "text",
                                        "sub_title": "Synthesizer:",
                                        "content": f"""<div style="font-size: 18px; margin-top:5px; margin-bottom:5px;">{synthesizer}</div>
                                                       <div style="font-size: 13px; margin-bottom:10px;">{seed_info()}</div>""",
                                    },
                                    # Second section
                                    {
                                        "type": "section",
                                        "title": "Summary Statistics (Real vs Synthetic Datasets)",
                                        "text": "This section compares the real and synthetic datasets using summary statistics.",
                                        "html_files": [summary_stats_filename], 
                                        "page_break": False,
                                    },
                                    # Third section
                                    {
                                        "type": "section",
                                        "title": "Downstream Model Training and Evaluation",        
                                        "content": f"""<div>
                                                    <p>Train on Real data and evaluate on Real data (TRTR) <br>
                                                    Train on Synthetic data evaluate on the Real dataset (TSTR).</p>
                                                    <div style="font-size:14px; margin-bottom: 20px;">
                                                        <b>Target: </b> {escape(str(target_name))}<br>
                                                        <b>Estimator: </b> {escape(str(estimator))}<br>
                                                    </div>
                                                    </div>
                                                    """,
                                        "html_files": [downstream_filename], 
                                        "page_break": False,
                                        
                                    },
              
                                    # Fourth section
                                    {
                                        "type": "section",
                                        "title": "SHAP Values Analysis",
                                        "content": f""" <div style='color:#0f736b; font-size:18px; margin-top:10px;'>
                                                        <b>SHAP Values Distribution per Feature</b></div>
                                                        <div><b>Model:</b>{estimator}</div><br>""",
                                        "image_files": [real_beeswarm_filename, syn_beeswarm_filename],
                                        
                                    },
              
                                   # Fifth section
                                    {
                                        "type": "section",
                                        "content": f"""<div style='color:#0f736b; font-size:18px; margin-top:20px; margin-bottom:20px;'>
                                                        <b>Feature Importance (Mean(|SHAP value|)</b><br>
                                                        </div>""",
                                        "image_files": [real_shap_bar_file, syn_shap_bar_file],
                                       
                                    }  
                                ] 
         
          utility_report = util.build_report(input_dir= utility_dir,sections= utility_report_sections, 
                                                        output_file= res_dir+"utility_report.html", title= "Synthetic Data Utility Report", 
                                                        title_style={"color": "#069174", "font-weight": "bold",})
          # filenames you want to KEEP
          utility_report_files = [summary_stats_filename, downstream_filename, real_beeswarm_filename, syn_beeswarm_filename, real_shap_bar_file, syn_shap_bar_file]
          util.clean_directory(utility_report_files, utility_dir)
                 
          utility_report_text_html = widgets.HTML(value=f"""<div style="color: #800080; font-weight:bold; font-size: 11pt; 
                                                          margin-top:15px;">Utility report saved in: {utility_report} </div>""")   
          display(utility_report_text_html)
         
               
# Hide the code cell
display(HTML("<style>.jp-Cell-inputWrapper {display: none; }</style>"))